# Step 1 ? Initial raw-data audit

This notebook reads competition source files without changing them. It reports file inventory, schemas, previews, missing values, exact duplicate rows within each file, memory usage, and observed code frequencies. No timestamp parsing, joins, cleaning, imputation, feature engineering, splits, or modelling are performed.

Taxi files are audited sequentially to limit memory use. The combined shape is calculated from compatible source schemas; no combined dataset is saved. Duplicate counts are exact **within each source file**; their sum does not include duplicates across different files.

In [1]:
from pathlib import Path
import gc
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("pandas:", pd.__version__)
print("Raw directory:", RAW)


pandas: 3.0.5
Raw directory: C:\Users\Shanujen\Desktop\urban-flow-datathon-2026\data\raw


## 1. Source inventory

Recursively inspect `data/raw/`. Ignore `__MACOSX` directories, Apple metadata entries beginning with `._`, and `.gitkeep` placeholders. Classify taxi and reference files using their actual filenames. Any unclassified files are shown for manual review. File sizes use decimal MB (1 MB = 1,000,000 bytes).

In [2]:
all_files = sorted(p for p in RAW.rglob("*") if p.is_file())
ignored = [p for p in all_files if "__MACOSX" in p.relative_to(RAW).parts or any(part.startswith("._") for part in p.relative_to(RAW).parts) or p.name == ".gitkeep"]
source_files = [p for p in all_files if p not in ignored]
supported = {".csv", ".parquet", ".xlsx", ".xls", ".json", ".jsonl", ".feather"}
usable = [p for p in source_files if p.suffix.lower() in supported]
taxi_files = [p for p in usable if p.name.lower().startswith("urban_flow_analytics_taxi_dataset_")]
reference_files = [p for p in usable if p not in taxi_files and any(word in p.stem.lower() for word in ("zone", "reference", "lookup"))]
unclassified = [p for p in source_files if p not in taxi_files + reference_files]
for path in source_files:
    role = "taxi" if path in taxi_files else "zone/reference" if path in reference_files else "UNCLASSIFIED"
    print(f"{path.relative_to(ROOT)} | {role} | {path.stat().st_size:,} bytes | {path.stat().st_size / 1e6:,.3f} MB")
print("Ignored metadata/placeholders:", [str(p.relative_to(RAW)) for p in ignored])
print("Taxi file count:", len(taxi_files))
print("Taxi data split across multiple files:", len(taxi_files) > 1)
print("Combined taxi source size (MB):", sum(p.stat().st_size for p in taxi_files) / 1e6)
print("Reference files:", [str(p.relative_to(ROOT)) for p in reference_files])
if not reference_files:
    print("MISSING: zone/reference dataset. Its shape and audit cannot be reported until supplied.")
if unclassified:
    print("WARNING: review unclassified sources:", [str(p.relative_to(ROOT)) for p in unclassified])
source_snapshot = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in source_files}


data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv | taxi | 429,662,745 bytes | 429.663 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv | taxi | 492,535,780 bytes | 492.536 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv | taxi | 462,646,895 bytes | 462.647 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv | taxi | 418,006,505 bytes | 418.007 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv | taxi | 383,928,038 bytes | 383.928 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv | taxi | 456,686,968 bytes | 456.687 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv | taxi | 477,118,702 bytes | 477.119 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv | taxi | 449,095,149 bytes | 449.095 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv | taxi | 460,962,917 bytes | 460.963 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv | taxi | 397,895,400 bytes | 397.895 MB
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-02

## 2. Read-only audit helpers

CSV files use `pandas.read_csv` with `parse_dates=False`; timestamps remain strings. Arrow-backed pandas columns reduce memory use without changing values. Standard pandas missing-value recognition is used at load time; no additional missing markers or cleaning rules are applied. Other formats use the matching pandas reader, with every Excel sheet audited separately.

Every dataset shows its exact source name, shape, complete column list and dtypes, first five rows, all-column missing counts, exact full-row duplicate count, and deep memory usage. Code frequency tables are selected only from columns actually present, using names containing code/flag or ending in `_id`, `_method`, or `_count`, plus low-cardinality text columns. These are observed values, not interpretations of undocumented codes.

In [3]:
def read_source(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        yield path.name, pd.read_csv(path, parse_dates=False, dtype_backend="pyarrow")
    elif suffix == ".parquet":
        yield path.name, pd.read_parquet(path)
    elif suffix in {".xlsx", ".xls"}:
        with pd.ExcelFile(path) as book:
            for sheet in book.sheet_names:
                yield f"{path.name} :: sheet={sheet}", pd.read_excel(book, sheet_name=sheet, parse_dates=False)
    elif suffix in {".json", ".jsonl"}:
        yield path.name, pd.read_json(path, lines=suffix == ".jsonl", convert_dates=False)
    elif suffix == ".feather":
        yield path.name, pd.read_feather(path)
    else:
        raise ValueError(f"Unsupported source: {path}")


def audit_frame(path, label, frame):
    display(Markdown(f"### `{label}`"))
    print("Exact source:", path.relative_to(ROOT))
    print("Shape:", frame.shape)
    print("All columns:", frame.columns.tolist())
    print("Data types:")
    print(frame.dtypes.to_string())
    print("First 5 rows:")
    display(frame.head(5))
    missing = frame.isna().sum()
    print("Missing-value counts (every column):")
    print(missing.to_string())
    duplicates = int(frame.duplicated().sum())
    memory_mb = float(frame.memory_usage(index=True, deep=True).sum() / 1e6)
    print("Exact duplicate rows within this dataset (after first occurrence):", duplicates)
    print("Total deep memory usage (MB):", memory_mb)
    categorical = []
    for column in frame.columns:
        name = str(column).lower()
        named_code = "code" in name or "flag" in name or name.endswith(("_id", "_method", "_count"))
        low_cardinality_text = (pd.api.types.is_string_dtype(frame[column].dtype) or pd.api.types.is_object_dtype(frame[column].dtype)) and frame[column].nunique(dropna=False) <= 30
        if named_code or low_cardinality_text:
            categorical.append(column)
    frequencies = {}
    print("Observed categorical/code columns:", categorical)
    for column in categorical:
        counts = frame[column].value_counts(dropna=False)
        frequencies[column] = counts
        print(f"\nValue counts: {column} (including missing; {len(counts)} distinct values)")
        print(counts.to_string())
    return {"source": str(path.relative_to(ROOT)), "dataset": label, "rows": len(frame),
            "columns": frame.columns.tolist(), "dtypes": frame.dtypes.astype(str).to_dict(),
            "missing": missing, "duplicates": duplicates, "memory_mb": memory_mb,
            "frequencies": frequencies}


## 3. Taxi source audits

Load and audit each real monthly taxi file separately. Release each dataframe after collecting small audit summaries. No source files or processed datasets are written.

In [4]:
taxi_audits = []
for path in taxi_files:
    print("Loading:", path.name, flush=True)
    for label, frame in read_source(path):
        taxi_audits.append(audit_frame(path, label, frame))
        del frame
    gc.collect()


Loading: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


### `Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv`

Exact source: data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Shape: (3970553, 20)
All columns: ['provider_code', 'pickup_timestamp', 'dropoff_timestamp', 'rider_count', 'distance_miles', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method', 'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment', 'toll_total', 'service_improvement_fee', 'charge_total', 'zone_congestion_fee', 'Airport_fee', 'congestion_relief_fee']
Data types:
provider_code               int64[pyarrow]
pickup_timestamp           string[pyarrow]
dropoff_timestamp          string[pyarrow]
rider_count                double[pyarrow]
distance_miles             double[pyarrow]
rate_class_id              double[pyarrow]
offline_record_flag        string[pyarrow]
origin_loc_id               int64[pyarrow]
dest_loc_id                 int64[pyarrow]
fare_settlement_method      int64[pyarrow]
base_fare                  double[pyarrow]
surcharge_misc             double[

,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee
0,1,2025-04-01 00:47:06,2025-04-01 01:13:25,1.0,9.5,1.0,N,138,230,1,38.7,11.0,0.5,11.65,6.94,1.0,69.79,2.5,1.75,0.75
1,2,2025-04-01 00:27:35,2025-04-01 00:38:19,2.0,3.77,1.0,N,138,92,1,17.0,6.0,0.5,4.9,0.0,1.0,31.15,0.0,1.75,0.0
2,2,2025-04-01 00:24:07,2025-04-01 00:35:12,1.0,5.41,1.0,N,132,130,1,22.6,1.0,0.5,5.37,0.0,1.0,32.22,0.0,1.75,0.0
3,1,2025-04-01 00:56:30,2025-04-01 01:00:49,2.0,0.6,1.0,N,79,4,1,6.5,4.25,0.5,2.45,0.0,1.0,14.7,2.5,0.0,0.75
4,2,2025-04-01 00:00:17,2025-04-01 00:16:19,1.0,0.43,1.0,N,161,229,2,4.4,1.0,0.5,0.0,0.0,1.0,10.15,2.5,0.0,0.75


Missing-value counts (every column):
provider_code                   0
pickup_timestamp                0
dropoff_timestamp               0
rider_count                745730
distance_miles                  0
rate_class_id              745730
offline_record_flag        745730
origin_loc_id                   0
dest_loc_id                     0
fare_settlement_method          0
base_fare                       0
surcharge_misc                  0
transit_tax                     0
driver_tip_payment              0
toll_total                      0
service_improvement_fee         0
charge_total                    0
zone_congestion_fee        745730
Airport_fee                745730
congestion_relief_fee           0
Exact duplicate rows within this dataset (after first occurrence): 0
Total deep memory usage (MB): 742.222373
Observed categorical/code columns: ['provider_code', 'rider_count', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method']

Value 

...

Missing-value counts (every column):
provider_code                    0
pickup_timestamp                 0
dropoff_timestamp                0
rider_count                1023317
distance_miles                   0
rate_class_id              1023317
offline_record_flag        1023317
origin_loc_id                    0
dest_loc_id                      0
fare_settlement_method           0
base_fare                        0
surcharge_misc                   0
transit_tax                      0
driver_tip_payment               0
toll_total                       0
service_improvement_fee          0
charge_total                     0
zone_congestion_fee        1023317
Airport_fee                1023317
congestion_relief_fee            0
Exact duplicate rows within this dataset (after first occurrence): 0
Total deep memory usage (MB): 635.402117
Observed categorical/code columns: ['provider_code', 'rider_count', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlem

### `Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv`

Exact source: data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv
Shape: (3952451, 20)
All columns: ['provider_code', 'pickup_timestamp', 'dropoff_timestamp', 'rider_count', 'distance_miles', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method', 'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment', 'toll_total', 'service_improvement_fee', 'charge_total', 'zone_congestion_fee', 'Airport_fee', 'congestion_relief_fee']
Data types:
provider_code               int64[pyarrow]
pickup_timestamp           string[pyarrow]
dropoff_timestamp          string[pyarrow]
rider_count                double[pyarrow]
distance_miles             double[pyarrow]
rate_class_id              double[pyarrow]
offline_record_flag        string[pyarrow]
origin_loc_id               int64[pyarrow]
dest_loc_id                 int64[pyarrow]
fare_settlement_method      int64[pyarrow]
base_fare                  double[pyarrow]
surcharge_misc             double[

,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.5,1.0,N,238,166,1,10.0,1.0,0.5,0.0,0.0,1.0,15.0,2.5,0.0,0.0
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2.0,0.88,1.0,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
3,2,2026-03-01 00:16:11,2026-03-01 00:28:20,1.0,1.76,1.0,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
4,2,2026-03-01 00:20:47,2026-03-01 00:30:44,2.0,1.57,1.0,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75


Missing-value counts (every column):
provider_code                   0
pickup_timestamp                0
dropoff_timestamp               0
rider_count                945748
distance_miles                  0
rate_class_id              945748
offline_record_flag        945748
origin_loc_id                   0
dest_loc_id                     0
fare_settlement_method          0
base_fare                       0
surcharge_misc                  0
transit_tax                     0
driver_tip_payment              0
toll_total                      0
service_improvement_fee         0
charge_total                    0
zone_congestion_fee        945748
Airport_fee                945748
congestion_relief_fee           0
Exact duplicate rows within this dataset (after first occurrence): 0
Total deep memory usage (MB): 738.769326
Observed categorical/code columns: ['provider_code', 'rider_count', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method']

Value 

## 4. Zone/reference audit

Audit supplied zone/reference datasets with the same read-only checks. If no reference file is present, record the missing input explicitly; do not download or invent a replacement.

In [5]:
reference_audits = []
for path in reference_files:
    for label, frame in read_source(path):
        reference_audits.append(audit_frame(path, label, frame))
        del frame
    gc.collect()
if not reference_audits:
    print("MISSING: zone/reference dataset; shape, columns, missing values, and duplicates are unavailable.")


MISSING: zone/reference dataset; shape, columns, missing values, and duplicates are unavailable.


## 5. Step 1 summary and Step 2 investigation candidates

Aggregate audit statistics only. A combined taxi shape is reported only when all files have the same ordered columns. Missing percentages use all taxi rows. Duplicate totals are explicitly scoped to within-file duplicates. Observed schema differences, missing fields, and unusual code values are investigation candidates; no values are changed or declared invalid without competition documentation.

In [6]:
print("Number of taxi source files:", len(taxi_files))
print("Combined taxi disk size: {:.3f} MB ({:.3f} GB)".format(sum(p.stat().st_size for p in taxi_files) / 1e6, sum(p.stat().st_size for p in taxi_files) / 1e9))
if taxi_audits:
    overview = pd.DataFrame([{key: result[key] for key in ("dataset", "rows", "duplicates", "memory_mb")} for result in taxi_audits])
    display(overview)
    total_rows = sum(result["rows"] for result in taxi_audits)
    same_schema = all(result["columns"] == taxi_audits[0]["columns"] for result in taxi_audits)
    print("Combined taxi shape (logical; no concatenation):", (total_rows, len(taxi_audits[0]["columns"])) if same_schema else "Different schemas: see per-file shapes")
    print("Sum of per-file deep memory usage (MB):", sum(result["memory_mb"] for result in taxi_audits))
    print("Exact within-file duplicate total:", sum(result["duplicates"] for result in taxi_audits))
    print("Cross-file duplicates: not evaluated; the above is not a global duplicate count.")
    if same_schema:
        missing = pd.concat([result["missing"] for result in taxi_audits], axis=1).sum(axis=1)
        summary = pd.DataFrame({"missing_count": missing.astype("int64"), "missing_percent": missing / total_rows * 100})
        print("Combined missing values (every column):")
        display(summary)
        print("Columns with missing values to investigate:", summary.index[summary.missing_count > 0].tolist())
        dtype_changes = {column: sorted({result["dtypes"][column] for result in taxi_audits}) for column in taxi_audits[0]["columns"]}
        print("Dtype differences across files:", {column: types for column, types in dtype_changes.items() if len(types) > 1})
        print("Code/count values to investigate against the data dictionary:")
        for column in taxi_audits[0]["frequencies"]:
            counts = pd.concat([result["frequencies"][column] for result in taxi_audits if column in result["frequencies"]])
            combined = counts.groupby(level=0, dropna=False).sum().sort_values(ascending=False)
            print(f"\n{column}: {len(combined)} distinct values; 15 most frequent")
            print(combined.head(15).to_string())
            if len(combined) <= 30:
                print("All observed values:", combined.index.tolist())
for result in reference_audits:
    print("Zone/reference shape:", result["dataset"], (result["rows"], len(result["columns"])))
    print("Zone/reference duplicates:", result["duplicates"])
if not reference_audits:
    print("Zone dataset shape: unavailable ? reference file missing.")
assert source_snapshot == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in source_files}, "Source file size or modification time changed during audit"
print("Source sizes and modification times unchanged.")
print("STOP: Step 1 complete for available inputs. No Step 2 operations performed.")


Number of taxi source files: 12
Combined taxi disk size: 5215.965 MB (5.216 GB)


,dataset,rows,duplicates,memory_mb
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,3970553,0,742.222373
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,4591845,0,858.239436
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,4322960,0,807.957436
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,3898963,1,728.722553
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,3574091,0,668.023365
5,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,4251015,0,794.543067
6,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,4428699,0,827.813978
7,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,4181444,0,781.562745
8,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,4305006,0,804.606282
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,3724889,0,696.155977


Combined taxi shape (logical; no concatenation): (48601782, 20)
Sum of per-file deep memory usage (MB): 9084.018655
Exact within-file duplicate total: 1
Cross-file duplicates: not evaluated; the above is not a global duplicate count.
Combined missing values (every column):


,missing_count,missing_percent
provider_code,0,0.000000
pickup_timestamp,0,0.000000
dropoff_timestamp,0,0.000000
rider_count,12405268,25.524307
distance_miles,0,0.000000
rate_class_id,12405268,25.524307
offline_record_flag,12405268,25.524307
origin_loc_id,0,0.000000
dest_loc_id,0,0.000000
fare_settlement_method,0,0.000000


Columns with missing values to investigate: ['rider_count', 'rate_class_id', 'offline_record_flag', 'zone_congestion_fee', 'Airport_fee']
Dtype differences across files: {}
Code/count values to investigate against the data dictionary:

provider_code: 4 distinct values; 15 most frequent
provider_code
2    38561069
1     9356276
7      642536
6       41901
All observed values: [2, 1, 7, 6]

rider_count: 11 distinct values; 15 most frequent
rider_count
1.0     28925980
<NA>    12405268
2.0      4874019
3.0      1143516
4.0       793921
0.0       231578
5.0       145773
6.0        81614
8.0           69
9.0           29
7.0           15
All observed values: [1.0, <NA>, 2.0, 3.0, 4.0, 0.0, 5.0, 6.0, 8.0, 9.0, 7.0]

rate_class_id: 8 distinct values; 15 most frequent
rate_class_id
1.0     33174103
<NA>    12405268
2.0      1257191
99.0     1030466
5.0       458398
3.0       157722
4.0       118586
6.0           48
All observed values: [1.0, <NA>, 2.0, 99.0, 5.0, 3.0, 4.0, 6.0]

offline_record

## Recorded findings ? available inputs only

- **Taxi sources:** 12 monthly CSV files, April 2025 through March 2026. Exact filenames and per-file shapes appear above.
- **Combined source size:** 5,215.965 MB (5.216 GB, decimal). Summed deep dataframe memory is 9,084.019 MB; files were loaded separately.
- **Logical combined taxi shape:** **48,601,782 rows ? 20 columns**. All monthly files have matching ordered columns and inferred dtypes. No combined dataframe or dataset file was created.
- **Zone/reference shape:** unavailable because no zone/reference dataset was present in `data/raw/`. This part of Step 1 remains pending until the file is supplied.
- **Missing values:** `rider_count`, `rate_class_id`, `offline_record_flag`, `zone_congestion_fee`, and `Airport_fee` each have **12,405,268 missing values (25.5243%)**. All other columns have zero missing values under the pandas CSV reader's standard missing-value recognition. Equal totals do not establish that the missing values occur on the same rows.
- **Exact within-file duplicates:** **1** in `Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv`; **0** in each of the other 11 taxi files. Cross-file duplicates were not evaluated; this is not a global duplicate count.

### Columns to investigate in Step 2 ? no rules applied

- The five columns with missing values above need a documented explanation before choosing any treatment.
- `rider_count` includes **231,578 zero values** and **113 values from 7 to 9**. Confirm the intended count semantics and applicable vehicle capacities.
- `rate_class_id` includes **1,030,466 values of 99**. Check the official code definitions.
- `fare_settlement_method` includes **12,405,268 values of 0** and only **2 values of 5**. The count for code 0 equals each of the five missing-value totals; whether these involve the same rows has **not** been tested.
- `provider_code` includes codes **6 and 7**, as well as 1 and 2. Confirm all codes using the competition data dictionary.
- `origin_loc_id` and `dest_loc_id` have **262 and 263 distinct values**, respectively. Reference coverage cannot be checked until the zone dataset is supplied; no zone join was performed.

These are investigation candidates, not confirmed invalid values. Source file sizes and modification times were unchanged at the end of execution. **Stop here: no timestamp parsing, cleaning, anomaly removal, feature engineering, data splitting, or modelling.**

## Step 2: Timestamp validation

This section is independent of Step 1's in-memory variables and can be run on its own. It validates timestamps only, reading the two timestamp columns in **200,000-row chunks**, one monthly file at a time. Source rows and files remain unchanged. Parsed series are temporary audit objects; invalid values are counted, not repaired or removed. No joins, trip durations, speed calculations, anomaly filtering, feature engineering, splitting, or modelling are performed.

In [7]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
from IPython.display import display

ts_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
ts_raw = ts_root / "data" / "raw"
ts_files = sorted(p for p in ts_raw.rglob("Urban_Flow_Analytics_Taxi_Dataset_*.csv")
                  if p.is_file() and "__MACOSX" not in p.relative_to(ts_raw).parts
                  and not any(part.startswith("._") for part in p.relative_to(ts_raw).parts))
assert len(ts_files) == 12, f"Expected 12 monthly taxi CSVs; found {len(ts_files)}"
ts_source_snapshot = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in ts_files}
# These names were observed in the source headers; verify every file before reading data.
ts_pickup = "pickup_timestamp"
ts_dropoff = "dropoff_timestamp"
for ts_path in ts_files:
    ts_columns = pd.read_csv(ts_path, nrows=0).columns.tolist()
    assert all(column in ts_columns for column in [ts_pickup, ts_dropoff]), ts_path.name
    assert re.fullmatch(r"Urban_Flow_Analytics_Taxi_Dataset_\d{4}-\d{2}\.csv", ts_path.name), ts_path.name
print("pandas version:", pd.__version__)
print("Actual timestamp columns:", ts_pickup, ts_dropoff)
print("Chunk size: 200,000 rows; only timestamp columns loaded")
print("Exact source filenames:")
for ts_path in ts_files:
    print(ts_path.relative_to(ts_root))


pandas version: 3.0.5
Actual timestamp columns: pickup_timestamp dropoff_timestamp
Chunk size: 200,000 rows; only timestamp columns loaded
Exact source filenames:
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
data\raw\Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


### Parsing, invalid counts, and file date ranges

Use `pd.to_datetime(..., errors="coerce", format="mixed", utc=False)` so parsing does not depend on the first row or chunk. Raw timestamp strings are read without CSV missing-marker conversion. Blank, missing-like, and unparseable strings become `NaT` in temporary parsed series and count as invalid; the source values are untouched. Minima and maxima refer only to successfully parsed timestamps.

Timezone status comes from parsed pandas dtypes, without assuming UTC or a local timezone. If offset-aware timestamps occur, monthly and daily counts use their displayed local calendar dates without converting timezone. Mixed incompatible timezone representations fail explicitly rather than being silently normalized.

The expected month is read from each source filename. Count valid pickups and dropoffs outside that calendar month separately from invalid timestamps. A dropoff just after month-end can be legitimate; these flags are observations, not cleaning rules.

In [8]:
ts_records = []
ts_daily_counts = Counter()
ts_timezone_kinds = {"pickup": set(), "dropoff": set()}
for ts_path in ts_files:
    ts_month = re.search(r"(\d{4}-\d{2})\.csv$", ts_path.name).group(1)
    ts_month_start = pd.Timestamp(ts_month + "-01")
    ts_next_month = ts_month_start + pd.offsets.MonthBegin(1)
    ts_record = {"source_file": ts_path.name, "expected_month": ts_month, "rows": 0}
    for ts_role in ["pickup", "dropoff"]:
        ts_record.update({f"invalid_{ts_role}": 0, f"earliest_{ts_role}": pd.NaT,
                          f"latest_{ts_role}": pd.NaT, f"{ts_role}_before_month": 0,
                          f"{ts_role}_after_month": 0})
    ts_file_zones = {"pickup": set(), "dropoff": set()}
    with pd.read_csv(ts_path, usecols=[ts_pickup, ts_dropoff], dtype="string",
                     keep_default_na=False, parse_dates=False, chunksize=200_000) as ts_reader:
        for ts_chunk in ts_reader:
            ts_record["rows"] += len(ts_chunk)
            for ts_role, ts_column in [("pickup", ts_pickup), ("dropoff", ts_dropoff)]:
                ts_parsed = pd.to_datetime(ts_chunk[ts_column], errors="coerce", format="mixed", utc=False)
                ts_record[f"invalid_{ts_role}"] += int(ts_parsed.isna().sum())
                if not ts_parsed.notna().any():
                    continue
                if not pd.api.types.is_datetime64_any_dtype(ts_parsed.dtype):
                    raise ValueError(f"Mixed timezone representations in {ts_path.name}: {ts_column}; no normalization applied")
                ts_tz = ts_parsed.dt.tz
                ts_zone = "timezone-naive" if ts_tz is None else f"timezone-aware ({ts_tz})"
                ts_file_zones[ts_role].add(ts_zone)
                ts_timezone_kinds[ts_role].add(ts_zone)
                if len(ts_timezone_kinds[ts_role]) > 1:
                    raise ValueError(f"Inconsistent timezone representations for {ts_role}; no normalization applied")
                ts_lo, ts_hi = ts_parsed.min(), ts_parsed.max()
                if pd.isna(ts_record[f"earliest_{ts_role}"]) or ts_lo < ts_record[f"earliest_{ts_role}"]:
                    ts_record[f"earliest_{ts_role}"] = ts_lo
                if pd.isna(ts_record[f"latest_{ts_role}"]) or ts_hi > ts_record[f"latest_{ts_role}"]:
                    ts_record[f"latest_{ts_role}"] = ts_hi
                ts_calendar = ts_parsed if ts_tz is None else ts_parsed.dt.tz_localize(None)
                ts_record[f"{ts_role}_before_month"] += int((ts_calendar < ts_month_start).sum())
                ts_record[f"{ts_role}_after_month"] += int((ts_calendar >= ts_next_month).sum())
                if ts_role == "pickup":
                    ts_counts = ts_calendar.dt.normalize().value_counts(dropna=True)
                    ts_daily_counts.update({day: int(count) for day, count in ts_counts.items()})
                del ts_parsed, ts_calendar
            del ts_chunk
    for ts_role in ["pickup", "dropoff"]:
        ts_record[f"{ts_role}_timezone"] = ", ".join(sorted(ts_file_zones[ts_role])) or "unknown (no valid timestamps)"
    ts_records.append(ts_record)
    print(f"Finished {ts_path.name}: {ts_record['rows']:,} rows; invalid pickup={ts_record['invalid_pickup']:,}, invalid dropoff={ts_record['invalid_dropoff']:,}", flush=True)

ts_file_summary = pd.DataFrame(ts_records)
print("Per-file timestamp validation (all rows and columns):")
print(ts_file_summary.to_string(index=False))


Finished Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 3,574,091 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 4,251,015 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 4,428,699 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 4,181,444 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 4,305,006 rows; invalid pickup=0, invalid dropoff=0
Finished Urban_Flow_Analytic

### Trips per pickup date across all 12 files

Sum daily pickup counts across chunks and source files. Each successfully parsed pickup contributes once, including dates outside the filename's month; no anomaly filtering is applied. Invalid pickups cannot be assigned a date and are reported separately. The table contains every observed pickup date (no artificial zero-count dates). Unique-date count is the number of rows in this aggregate table. This table is an audit summary only; no date feature is added to the taxi data.

In [9]:
ts_trips_per_day = pd.DataFrame(sorted(ts_daily_counts.items()), columns=["pickup_date", "trip_count"])
print("Number of unique valid pickup dates:", len(ts_trips_per_day))
print("Trips per day across all source files:")
print(ts_trips_per_day.to_string(index=False))
print("Rows with invalid pickup timestamp (unassigned to a day):", int(ts_file_summary["invalid_pickup"].sum()))
assert int(ts_trips_per_day["trip_count"].sum()) + int(ts_file_summary["invalid_pickup"].sum()) == int(ts_file_summary["rows"].sum())
print("PASS: daily counts plus invalid pickups equal total source rows.")


Number of unique valid pickup dates: 369
Trips per day across all source files:
pickup_date  trip_count
 2008-12-31           2
 2009-01-01           6
 2025-03-31           4
 2025-04-01      128955
 2025-04-02      137393
 2025-04-03      138411
 2025-04-04      135084
 2025-04-05      153237
 2025-04-06      122921
 2025-04-07      115829
 2025-04-08      131574
 2025-04-09      137646
 2025-04-10      147648
 2025-04-11      154758
 2025-04-12      149391
 2025-04-13      109780
 2025-04-14       99795
 2025-04-15      122082
 2025-04-16      133969
 2025-04-17      135522
 2025-04-18      125581
 2025-04-19      134049
 2025-04-20      115504
 2025-04-21      107881
 2025-04-22      124843
 2025-04-23      135843
 2025-04-24      148387
 2025-04-25      147589
 2025-04-26      158506
 2025-04-27      141509
 2025-04-28      110105
 2025-04-29      126466
 2025-04-30      140308
 2025-05-01      157143
 2025-05-02      147086
 2025-05-03      164004
 2025-05-04      142682
 2025-05

### Overall timestamp summary and filename-month flags

Report the global extrema and invalid totals, then list each file with out-of-month timestamps. A timestamp can parse successfully and still have a suspicious date. All such rows remain included. These checks do not establish whether a trip duration, location, or other value is valid.

In [10]:
ts_overall = {
    "source_files": len(ts_files),
    "total_rows": int(ts_file_summary["rows"].sum()),
    "invalid_pickup": int(ts_file_summary["invalid_pickup"].sum()),
    "invalid_dropoff": int(ts_file_summary["invalid_dropoff"].sum()),
    "earliest_pickup": ts_file_summary["earliest_pickup"].min(),
    "latest_pickup": ts_file_summary["latest_pickup"].max(),
    "earliest_dropoff": ts_file_summary["earliest_dropoff"].min(),
    "latest_dropoff": ts_file_summary["latest_dropoff"].max(),
    "unique_pickup_dates": len(ts_trips_per_day),
}
for ts_key, ts_value in ts_overall.items():
    print(f"{ts_key}: {ts_value}")
print("Timezone status:", {role: sorted(kinds) for role, kinds in ts_timezone_kinds.items()})
ts_range_columns = ["pickup_before_month", "pickup_after_month", "dropoff_before_month", "dropoff_after_month"]
ts_range_flags = ts_file_summary.loc[ts_file_summary[ts_range_columns].sum(axis=1) > 0,
    ["source_file", "expected_month"] + ts_range_columns]
print("Files with valid timestamps outside the filename month:")
print(ts_range_flags.to_string(index=False) if len(ts_range_flags) else "None")
print("Totals outside filename month:")
print(ts_file_summary[ts_range_columns].sum().to_string())
assert ts_source_snapshot == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in ts_files}
print("PASS: source file sizes and modification times unchanged.")
print("STOP: Step 2 timestamp validation only; no rows changed or removed.")


source_files: 12
total_rows: 48601782
invalid_pickup: 0
invalid_dropoff: 0
earliest_pickup: 2008-12-31 23:03:20
latest_pickup: 2026-04-01 00:06:25
earliest_dropoff: 2008-12-31 23:32:25
latest_dropoff: 2026-04-02 16:03:47
unique_pickup_dates: 369
Timezone status: {'pickup': ['timezone-naive'], 'dropoff': ['timezone-naive']}
Files with valid timestamps outside the filename month:
                                  source_file expected_month  pickup_before_month  pickup_after_month  dropoff_before_month  dropoff_after_month
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv        2025-04                    4                   3                     1                 1102
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv        2025-05                   21                   3                     9                 2930
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv        2025-06                   20                   0                     7                  745
Urban_Flow_Analytics_Taxi_Dataset_2025-

### Step 2 recorded findings

- **Coverage:** all 12 monthly taxi CSVs; **48,601,782 rows**, processed sequentially in chunks of at most 200,000 rows. Actual fields: `pickup_timestamp` and `dropoff_timestamp`.
- **Invalid pickup timestamps:** **0**. **Invalid dropoff timestamps:** **0**. Successful parsing does not establish that the recorded date is correct.
- **Pickup range:** **2008-12-31 23:03:20** through **2026-04-01 00:06:25**.
- **Dropoff range:** **2008-12-31 23:32:25** through **2026-04-02 16:03:47**.
- **Timezone status:** both columns are **timezone-naive** in every chunk/file. The source strings do not establish a timezone, and none was assigned.
- **Unique pickup dates:** **369**, including suspicious dates. The complete trips-per-day table above sums to **48,601,782**, matching the source row count.
- **Filename-month mismatches:** **170 pickups** (152 before, 18 after) and **19,538 dropoffs** (44 before, 19,494 after). These are separate field counts, not a count of distinct affected trips. All source rows remain included.

All filenames below have the prefix `Urban_Flow_Analytics_Taxi_Dataset_` and suffix `.csv`. Full filenames and exact per-file minima/maxima are printed above.

| File month | Pickups outside filename month | Dropoffs outside filename month | Date-range observation |
|---|---:|---:|---|
| 2025-04 | 7 | 1,103 | March 31 through May 1 |
| 2025-05 | 24 | 2,939 | Earliest pickup/dropoff in January 2009; dropoffs through June 4 |
| 2025-06 | 20 | 752 | Pickups begin May 31; dropoffs through July 1 |
| 2025-07 | 7 | 1,217 | Earliest pickup/dropoff in January 2009; dropoffs through August 3 |
| 2025-08 | 17 | 1,493 | Earliest pickup/dropoff in January 2009; dropoffs through September 2 |
| 2025-09 | 7 | 986 | Pickups begin August 31; dropoffs through October 3 |
| 2025-10 | 13 | 3,989 | Pickups begin September 30; dropoffs through November 3 |
| 2025-11 | 24 | 809 | Earliest pickup/dropoff in December 2008; dropoffs through December 1 |
| 2025-12 | 9 | 778 | Pickups begin November 30; dropoffs through January 5, 2026 |
| 2026-01 | 7 | 2,284 | Pickups begin December 31; dropoffs through February 1 |
| 2026-02 | 16 | 2,338 | Pickups begin January 31; dropoffs through March 1 |
| 2026-03 | 19 | 850 | Earliest pickup in December 2008 and dropoff in January 2009; dropoffs through April 2 |

Crossing a month boundary can be legitimate, especially for dropoffs. Historical dates and multi-day month spillovers deserve investigation, but this audit does not calculate durations or impose anomaly rules. No rows were removed, corrected, or saved as cleaned data. Source sizes and modification times remained unchanged. **Stop after Step 2.**

## Step 3: Zone/reference audit and join preparation

This section audits the newly supplied reference and checks key coverage without performing a taxi-zone join. It runs independently of earlier in-memory variables and preserves the earlier audit outputs as historical observations. Source files and taxi rows are not modified. Only `origin_loc_id` and `dest_loc_id` are read from taxi CSVs, sequentially in chunks of 200,000 rows.

No rows are removed, imputed, or saved as cleaned/joined data. No timestamp processing, duration/speed calculations, anomaly filtering, feature engineering, splitting, or modelling is performed here.

In [11]:
from pathlib import Path
from collections import Counter
import pandas as pd

z_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
z_raw_dir = z_root / "data" / "raw"
z_sources = sorted(p for p in z_raw_dir.rglob("*") if p.is_file()
                   and "__MACOSX" not in p.relative_to(z_raw_dir).parts
                   and not any(part.startswith("._") for part in p.relative_to(z_raw_dir).parts)
                   and p.name != ".gitkeep")
z_candidates = [p for p in z_sources if p.suffix.lower() == ".csv" and "zone" in p.stem.lower()]
assert len(z_candidates) == 1, f"Expected one zone CSV; review candidates: {z_candidates}"
z_path = z_candidates[0]
z_taxi_files = [p for p in z_sources if p.suffix.lower() == ".csv" and p.name.startswith("Urban_Flow_Analytics_Taxi_Dataset_")]
assert len(z_taxi_files) == 12, f"Expected 12 taxi CSVs; found {len(z_taxi_files)}"
z_snapshot = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in [z_path, *z_taxi_files]}
z_reference = pd.read_csv(z_path, parse_dates=False)
print("Exact reference filename:", z_path.name)
print("Source path:", z_path.relative_to(z_root))
print("Shape:", z_reference.shape)
print("All columns:", z_reference.columns.tolist())
print("Data types:")
print(z_reference.dtypes.to_string())
print("First 10 rows:")
print(z_reference.head(10).to_string(index=False))
print("Missing-value counts (standard pandas CSV missing markers):")
print(z_reference.isna().sum().to_string())
print("Duplicate rows after the first occurrence:", int(z_reference.duplicated().sum()))


Exact reference filename: Urban_Flow_Analytics_Zone_Dataset (1).csv
Source path: data\raw\Urban_Flow_Analytics_Zone_Dataset (1).csv
Shape: (265, 4)
All columns: ['loc_id', 'borough_name', 'zone_name', 'service_zone']
Data types:
loc_id          int64
borough_name      str
zone_name         str
service_zone      str
First 10 rows:
 loc_id  borough_name               zone_name service_zone
      1           EWR          Newark Airport          EWR
      2        Queens             Jamaica Bay    Boro Zone
      3         Bronx Allerton/Pelham Gardens    Boro Zone
      4     Manhattan           Alphabet City  Yellow Zone
      5 Staten Island           Arden Heights    Boro Zone
      6 Staten Island Arrochar/Fort Wadsworth    Boro Zone
      7        Queens                 Astoria    Boro Zone
      8        Queens            Astoria Park    Boro Zone
      9        Queens              Auburndale    Boro Zone
     10        Queens            Baisley Park    Boro Zone
Missing-value count

### Reference key and readable fields

The actual reference column `loc_id` is the candidate key for the actual taxi columns `origin_loc_id` and `dest_loc_id`. The readable fields present are `borough_name`, `zone_name`, and `service_zone`; no additional field meanings or code definitions are inferred. Compare observed dtypes and exact ID membership without casting, normalizing, or rewriting source values.

Check key nulls, uniqueness, duplicate IDs, and range before any future join. Duplicate reference keys would cause row multiplication in a join and require review. For labels that pandas reads as missing, show the corresponding literal CSV strings separately, with missing-marker conversion disabled; this is an inspection view, not an imputation or edit.

In [12]:
z_key = "loc_id"
z_readable = ["borough_name", "zone_name", "service_zone"]
assert all(column in z_reference.columns for column in [z_key, *z_readable])
for z_taxi_path in z_taxi_files:
    z_header = pd.read_csv(z_taxi_path, nrows=0).columns
    assert all(column in z_header for column in ["origin_loc_id", "dest_loc_id"]), z_taxi_path.name
z_ids = set(z_reference[z_key].dropna())
z_duplicate_key_mask = z_reference[z_key].notna() & z_reference[z_key].duplicated(keep=False)
z_duplicate_ids = z_reference.loc[z_duplicate_key_mask, z_key].unique().tolist()
print("Candidate mappings: origin_loc_id -> loc_id; dest_loc_id -> loc_id")
print("Readable location columns:", z_readable)
print("Reference key dtype:", z_reference[z_key].dtype)
print("Unique non-missing zone IDs:", z_reference[z_key].nunique(dropna=True))
print("Missing zone IDs:", int(z_reference[z_key].isna().sum()))
print("All zone IDs unique:", z_reference[z_key].is_unique)
print("Minimum zone ID:", z_reference[z_key].min())
print("Maximum zone ID:", z_reference[z_key].max())
print("Duplicate zone IDs:", z_duplicate_ids)
print("Rows involved in duplicate IDs:", int(z_duplicate_key_mask.sum()))
if z_duplicate_ids:
    print(z_reference.loc[z_duplicate_key_mask].to_string(index=False))
z_incomplete_labels = z_reference[z_readable].isna().any(axis=1)
print("Reference rows with missing readable fields:")
print(z_reference.loc[z_incomplete_labels].to_string(index=False))
z_literal_reference = pd.read_csv(z_path, dtype="string", keep_default_na=False, parse_dates=False)
print("Literal CSV text for the same rows (N/A preserved as text):")
print(z_literal_reference.loc[z_incomplete_labels].to_string(index=False))


Candidate mappings: origin_loc_id -> loc_id; dest_loc_id -> loc_id
Readable location columns: ['borough_name', 'zone_name', 'service_zone']
Reference key dtype: int64
Unique non-missing zone IDs: 265
Missing zone IDs: 0
All zone IDs unique: True
Minimum zone ID: 1
Maximum zone ID: 265
Duplicate zone IDs: []
Rows involved in duplicate IDs: 0
Reference rows with missing readable fields:
 loc_id borough_name      zone_name service_zone
    264      Unknown            NaN          NaN
    265          NaN Outside of NYC          NaN
Literal CSV text for the same rows (N/A preserved as text):
loc_id borough_name      zone_name service_zone
   264      Unknown            N/A          N/A
   265          N/A Outside of NYC          N/A


### Taxi-to-reference coverage across all monthly files

Count distinct non-missing origin and destination IDs across all chunks, and count rows whose ID is absent from the reference. Missing taxi IDs, if present, count as unmatched and are also reported separately. All row percentages use the full source row count as denominator, not the number of distinct IDs. Invalid or unmatched values are only counted; no taxi rows are changed or excluded.

A matched ID does not guarantee a complete readable label. Report usage of reference rows with missing readable fields separately. This distinguishes absent keys from existing keys with incomplete labels, without joining any taxi rows.

In [13]:
z_origin_counts = Counter()
z_destination_counts = Counter()
z_origin_unmatched_counts = Counter()
z_destination_unmatched_counts = Counter()
z_file_records = []
z_taxi_dtypes = {"origin_loc_id": set(), "dest_loc_id": set()}
for z_taxi_path in z_taxi_files:
    z_record = {"source_file": z_taxi_path.name, "rows": 0,
                "unmatched_origin_rows": 0, "unmatched_destination_rows": 0,
                "missing_origin_rows": 0, "missing_destination_rows": 0}
    with pd.read_csv(z_taxi_path, usecols=["origin_loc_id", "dest_loc_id"],
                     parse_dates=False, chunksize=200_000) as z_reader:
        for z_chunk in z_reader:
            z_record["rows"] += len(z_chunk)
            for z_column, z_role, z_counts, z_unmatched_counts in [
                ("origin_loc_id", "origin", z_origin_counts, z_origin_unmatched_counts),
                ("dest_loc_id", "destination", z_destination_counts, z_destination_unmatched_counts),
            ]:
                z_values = z_chunk[z_column]
                z_taxi_dtypes[z_column].add(str(z_values.dtype))
                z_counts.update({key: int(value) for key, value in z_values.value_counts(dropna=True).items()})
                z_unmatched = ~z_values.isin(z_ids)
                z_record[f"unmatched_{z_role}_rows"] += int(z_unmatched.sum())
                z_record[f"missing_{z_role}_rows"] += int(z_values.isna().sum())
                z_unmatched_counts.update({key: int(value) for key, value in z_values[z_unmatched].value_counts(dropna=True).items()})
            del z_chunk, z_values, z_unmatched
    z_record["unmatched_origin_percent"] = 100 * z_record["unmatched_origin_rows"] / z_record["rows"] if z_record["rows"] else float("nan")
    z_record["unmatched_destination_percent"] = 100 * z_record["unmatched_destination_rows"] / z_record["rows"] if z_record["rows"] else float("nan")
    z_file_records.append(z_record)
    print(f"Finished {z_taxi_path.name}: {z_record['rows']:,} rows; unmatched origin={z_record['unmatched_origin_rows']:,}, unmatched destination={z_record['unmatched_destination_rows']:,}", flush=True)
z_file_coverage = pd.DataFrame(z_file_records)
print("Observed taxi ID dtypes:", {column: sorted(types) for column, types in z_taxi_dtypes.items()})
print("Per-file key coverage:")
print(z_file_coverage.to_string(index=False))


Finished Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 3,574,091 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 4,251,015 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 4,428,699 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 4,181,444 rows; unmatched origin=0, unmatched destination=0
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 4,305,006 rows; 

### Overall coverage and join readiness

Aggregate the counters, show every unmatched non-missing ID and its frequency, and distinguish key coverage from label completeness. Reference-only IDs are reported as observations, not defects. A future join should be a left join using the verified key with `validate="many_to_one"` to prevent unexpected row multiplication; no such join is executed in this step.

In [14]:
z_total_rows = int(z_file_coverage["rows"].sum())
print("Zone dataset shape:", z_reference.shape)
print("Zone ID column:", z_key)
print("Readable location columns:", z_readable)
print("Unique zone IDs:", len(z_ids))
print("Duplicate zone IDs:", z_duplicate_ids)
print("Total taxi rows checked:", z_total_rows)
print("Unique non-missing origin IDs:", len(z_origin_counts))
print("Unique non-missing destination IDs:", len(z_destination_counts))
for z_role, z_counts, z_unmatched_counts in [
    ("origin", z_origin_counts, z_origin_unmatched_counts),
    ("destination", z_destination_counts, z_destination_unmatched_counts),
]:
    z_missing_total = int(z_file_coverage[f"missing_{z_role}_rows"].sum())
    z_unmatched_total = int(z_file_coverage[f"unmatched_{z_role}_rows"].sum())
    print(f"Unmatched {z_role} IDs and row counts:", dict(z_unmatched_counts))
    print(f"Missing {z_role} IDs (rows):", z_missing_total)
    print(f"Unmatched {z_role} rows:", z_unmatched_total)
    print(f"Percentage of all taxi rows with unmatched {z_role} IDs: {100 * z_unmatched_total / z_total_rows:.6f}%")
    print(f"Reference IDs not observed as {z_role}:", sorted(z_ids - set(z_counts)))
    assert sum(z_counts.values()) + z_missing_total == z_total_rows
    assert sum(z_unmatched_counts.values()) + z_missing_total == z_unmatched_total
z_label_usage = z_reference.loc[z_incomplete_labels, [z_key, *z_readable]].copy()
z_label_usage["origin_rows"] = z_label_usage[z_key].map(z_origin_counts).fillna(0).astype("int64")
z_label_usage["destination_rows"] = z_label_usage[z_key].map(z_destination_counts).fillna(0).astype("int64")
z_label_usage["origin_percent"] = z_label_usage["origin_rows"] / z_total_rows * 100
z_label_usage["destination_percent"] = z_label_usage["destination_rows"] / z_total_rows * 100
print("Matched reference IDs with incomplete readable labels (aggregate report only):")
print(z_label_usage.to_string(index=False))
print("Key unique and non-missing:", z_reference[z_key].is_unique and z_reference[z_key].notna().all())
assert z_snapshot == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in [z_path, *z_taxi_files]}
print("PASS: ID counts reconcile to all taxi rows; source sizes and modification times unchanged.")
print("STOP: Step 3 audit and join coverage only. No taxi-zone join or source-data writes performed.")


Zone dataset shape: (265, 4)
Zone ID column: loc_id
Readable location columns: ['borough_name', 'zone_name', 'service_zone']
Unique zone IDs: 265
Duplicate zone IDs: []
Total taxi rows checked: 48601782
Unique non-missing origin IDs: 262
Unique non-missing destination IDs: 263
Unmatched origin IDs and row counts: {}
Missing origin IDs (rows): 0
Unmatched origin rows: 0
Percentage of all taxi rows with unmatched origin IDs: 0.000000%
Reference IDs not observed as origin: [103, 104, 110]
Unmatched destination IDs and row counts: {}
Missing destination IDs (rows): 0
Unmatched destination rows: 0
Percentage of all taxi rows with unmatched destination IDs: 0.000000%
Reference IDs not observed as destination: [103, 104]
Matched reference IDs with incomplete readable labels (aggregate report only):
 loc_id borough_name      zone_name service_zone  origin_rows  destination_rows  origin_percent  destination_percent
    264      Unknown            NaN          NaN        70293             89644 

### Step 3 recorded findings

- **Exact reference file:** `Urban_Flow_Analytics_Zone_Dataset (1).csv`.
- **Shape:** **265 rows x 4 columns**. Columns are `loc_id`, `borough_name`, `zone_name`, and `service_zone`.
- **Join key:** `loc_id` for both taxi `origin_loc_id` and `dest_loc_id`. All three fields have observed `int64` dtype, and every taxi ID is present in the reference. No key coercion or join was performed.
- **Readable fields:** `borough_name`, `zone_name`, and `service_zone`, as named in the actual source file.
- **Reference integrity:** **265 unique IDs**, minimum **1**, maximum **265**; **0 missing IDs**, **0 duplicate IDs**, and **0 duplicate rows**. The key supports a future many-to-one join without reference-key row multiplication.
- **Taxi coverage:** all **12 files / 48,601,782 rows** checked in chunks. There are **262 unique origin IDs** and **263 unique destination IDs**, with no missing taxi IDs.
- **Unmatched origin IDs:** none; **0 rows (0%)**. **Unmatched destination IDs:** none; **0 rows (0%)**. Every monthly file has zero unmatched rows for both fields.
- **Reference IDs not observed:** origins: **103, 104, 110**; destinations: **103, 104**. Non-use alone is not a reference defect.

#### Reference issues requiring interpretation before later use

Standard pandas CSV loading reports **1 missing `borough_name`**, **1 missing `zone_name`**, and **2 missing `service_zone` values**. The literal file contains `N/A` at those positions; no missing values were filled or modified.

| loc_id | Actual reference labels | Origin rows (% of all taxi rows) | Destination rows (% of all taxi rows) |
|---|---|---:|---:|
| 264 | `borough_name=Unknown`; `zone_name=N/A`; `service_zone=N/A` | 70,293 (0.144630%) | 89,644 (0.184446%) |
| 265 | `borough_name=N/A`; `zone_name=Outside of NYC`; `service_zone=N/A` | 22,560 (0.046418%) | 227,614 (0.468324%) |

These IDs match the reference but do not supply complete readable location labels. Their exact meanings or treatment should not be inferred beyond the source labels. Origin and destination counts are separate and must not be added as a count of distinct affected trips.

Earlier Step 1 statements that the reference was missing describe the inputs available at that earlier audit; the reference is now supplied and audited here. Counts reconcile to all source rows, and source sizes and modification times remain unchanged. **Stop after Step 3: no taxi-zone join, cleaning, row removal, imputation, or later processing was performed.**

## Step 4: Audit-calculation fields

Create temporary audit fields in 200,000-row chunks, without concatenating the monthly data. Steps 1-3 remain unchanged. Only the five required source columns are loaded; every source row contributes to this step. No fare, distance, rider-count, duration, speed, or timestamp filtering is applied, and no values are imputed.

| Field | Calculation | Use in a later audit |
|---|---|---|
| `trip_duration_minutes` | Dropoff minus pickup, in minutes | Inspect recorded elapsed time |
| `speed_mph` | `distance_miles / (trip_duration_minutes / 60)`, only for positive duration | Inspect implied speed; undefined for non-positive duration |
| `pickup_date` | Pickup calendar date | Inspect date coverage |
| `pickup_hour` | Pickup hour, 0-23 | Inspect time-of-day patterns |
| `day_of_week` | Readable pickup weekday name | Inspect weekday patterns |
| `month` | Pickup calendar month number, 1-12 | Inspect month patterns; this is not a year-month key |
| `weekend` | Saturday or Sunday | Compare weekday/weekend coverage |
| `route_id` | Original origin ID + `-` + original destination ID | Reproducible directional route label |

Timestamp parsing uses `errors="coerce"` and `format="mixed"`, as in Step 2, without assigning a timezone. Invalid timestamps produce missing derived calendar/duration values. Missing inputs remain missing. These calculations prepare an audit view only; no anomaly counts, thresholds, cleaning rules, or downstream processing are introduced.

In [ ]:
from pathlib import Path
from collections import Counter
import tempfile
import math
import numpy as np
import pandas as pd

a_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
a_files = sorted(p for p in (a_root / "data" / "raw").rglob("Urban_Flow_Analytics_Taxi_Dataset_*.csv")
                 if p.is_file() and "__MACOSX" not in p.relative_to(a_root).parts
                 and not any(part.startswith("._") for part in p.relative_to(a_root).parts))
assert len(a_files) == 12, f"Expected 12 taxi files; found {len(a_files)}"
# Names verified from the actual CSV headers, not inferred alternatives.
a_columns = {"pickup": "pickup_timestamp", "dropoff": "dropoff_timestamp", "distance": "distance_miles",
             "origin": "origin_loc_id", "destination": "dest_loc_id"}
for a_path in a_files:
    a_header = pd.read_csv(a_path, nrows=0).columns.tolist()
    a_missing = set(a_columns.values()) - set(a_header)
    if a_missing:
        raise ValueError(f"Essential columns missing in {a_path.name}: {sorted(a_missing)}")
a_snapshot = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in a_files}
print("Verified source columns:", a_columns)
print("Files:")
for a_path in a_files:
    print(a_path.name)

def a_derive(chunk):
    if not pd.api.types.is_numeric_dtype(chunk[a_columns["distance"]]):
        raise TypeError("distance_miles is not numeric; stop without coercing source values")
    pickup = pd.to_datetime(chunk[a_columns["pickup"]], errors="coerce", format="mixed", utc=False)
    dropoff = pd.to_datetime(chunk[a_columns["dropoff"]], errors="coerce", format="mixed", utc=False)
    if not pd.api.types.is_datetime64_any_dtype(pickup.dtype) or not pd.api.types.is_datetime64_any_dtype(dropoff.dtype):
        raise TypeError("Unexpected timestamp dtype; stop without timezone normalization")
    if pickup.dt.tz is not None or dropoff.dt.tz is not None:
        raise TypeError("Timezone-aware timestamps differ from Step 2; review before calculating")
    enriched = chunk.copy()
    enriched["trip_duration_minutes"] = (dropoff - pickup).dt.total_seconds() / 60
    enriched["speed_mph"] = np.nan
    positive = enriched["trip_duration_minutes"] > 0
    # Division is executed only on positive-duration records.
    enriched.loc[positive, "speed_mph"] = enriched.loc[positive, a_columns["distance"]] / (enriched.loc[positive, "trip_duration_minutes"] / 60)
    enriched["pickup_date"] = pickup.dt.date
    enriched["pickup_hour"] = pickup.dt.hour.astype("Int64")
    enriched["day_of_week"] = pickup.dt.day_name()
    enriched["month"] = pickup.dt.month.astype("Int64")
    enriched["weekend"] = pickup.dt.dayofweek.astype("Int64") >= 5
    enriched["route_id"] = chunk[a_columns["origin"]].astype("string") + "-" + chunk[a_columns["destination"]].astype("string")
    assert enriched.loc[~positive, "speed_mph"].isna().all()
    assert len(enriched) == len(chunk)
    return enriched

Verified source columns: {'pickup': 'pickup_timestamp', 'dropoff': 'dropoff_timestamp', 'distance': 'distance_miles', 'origin': 'origin_loc_id', 'destination': 'dest_loc_id'}
Files:
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


### Incremental descriptive statistics and exact medians

Accumulate counts, sums, minima, maxima, weekday counts, and weekend counts across chunks. Duration statistics include all defined durations, including zero and negative values. Speed statistics include all defined speeds on positive-duration rows, including zero/negative distances and extreme results; no plausibility threshold is used.

For **exact** medians, append only the two numeric calculation vectors to temporary binary files, excluding undefined values only from their respective statistics. Partition each disk-backed vector in place, then remove the temporary files automatically. This requires approximately 0.8 GB of temporary disk space, stores no enriched taxi rows, and creates no persistent dataset or Parquet output. Only the current chunk, small counters, and ten sample rows are retained in the notebook process.

The ten validation examples are purposive, not a statistically representative sample: six file-start examples spread across the year, a zero-duration example if present, and rows attaining minimum duration, maximum duration, and maximum speed. These demonstrate calculation behavior without counting anomalies or removing records.

In [16]:
a_stats = {field: {"n": 0, "min": math.inf, "max": -math.inf, "chunk_sums": []}
           for field in ["trip_duration_minutes", "speed_mph"]}
a_weekdays = Counter()
a_weekends = Counter()
a_dtypes = {column: set() for column in a_columns.values()}
a_field_dtypes = {}
a_hour_min, a_hour_max = None, None
a_rows = 0
a_file_rows = []
a_samples = {}
a_sample_file_indices = {0, 2, 4, 6, 8, 11}
a_extrema = {"minimum duration": math.inf, "maximum duration": -math.inf, "maximum speed": -math.inf}

def a_sample_record(frame, index, filename, reason):
    row = frame.loc[index].copy()
    row["source_file"] = filename
    row["source_row_1based"] = int(index) + 1
    row["sample_reason"] = reason
    return row

def a_exact_median(path, n):
    if n == 0:
        return float("nan")
    vector = np.memmap(path, dtype="float64", mode="r+", shape=(n,))
    try:
        middle = n // 2
        if n % 2:
            vector.partition(middle)
            result = float(vector[middle])
        else:
            vector.partition((middle - 1, middle))
            result = float(vector[middle - 1]) / 2 + float(vector[middle]) / 2
        vector.flush()
        return result
    finally:
        vector._mmap.close()

with tempfile.TemporaryDirectory(prefix="urban_flow_step4_medians_") as a_tmp:
    a_temp_paths = {field: Path(a_tmp) / f"{field}.bin" for field in a_stats}
    a_handles = {field: path.open("wb") for field, path in a_temp_paths.items()}
    try:
        for a_file_index, a_path in enumerate(a_files):
            a_this_file_rows = 0
            with pd.read_csv(a_path, usecols=list(a_columns.values()), parse_dates=False, chunksize=200_000) as a_reader:
                for a_chunk in a_reader:
                    for column in a_columns.values():
                        a_dtypes[column].add(str(a_chunk[column].dtype))
                    a_enriched = a_derive(a_chunk)
                    for field in a_enriched.columns:
                        if field not in a_columns.values():
                            a_field_dtypes.setdefault(field, set()).add(str(a_enriched[field].dtype))
                    a_rows += len(a_enriched)
                    a_this_file_rows += len(a_enriched)
                    for field, stats in a_stats.items():
                        values = a_enriched[field].dropna().to_numpy(dtype="float64")
                        if len(values):
                            stats["n"] += len(values)
                            stats["min"] = min(stats["min"], float(values.min()))
                            stats["max"] = max(stats["max"], float(values.max()))
                            stats["chunk_sums"].append(float(values.sum()))
                            values.tofile(a_handles[field])
                        del values
                    valid_hours = a_enriched["pickup_hour"].dropna()
                    if len(valid_hours):
                        lo, hi = int(valid_hours.min()), int(valid_hours.max())
                        a_hour_min = lo if a_hour_min is None else min(a_hour_min, lo)
                        a_hour_max = hi if a_hour_max is None else max(a_hour_max, hi)
                    a_weekdays.update({str(key): int(value) for key, value in a_enriched["day_of_week"].value_counts(dropna=False).items()})
                    a_weekends.update({"missing" if pd.isna(key) else "weekend" if bool(key) else "weekday": int(value)
                                       for key, value in a_enriched["weekend"].value_counts(dropna=False).items()})
                    sample_key = f"file {a_file_index}"
                    if a_file_index in a_sample_file_indices and sample_key not in a_samples:
                        a_samples[sample_key] = a_sample_record(a_enriched, a_enriched.index[0], a_path.name, "file-start example")
                    if "zero duration" not in a_samples:
                        zero = a_enriched["trip_duration_minutes"].eq(0)
                        if zero.any():
                            a_samples["zero duration"] = a_sample_record(a_enriched, zero[zero].index[0], a_path.name, "zero duration")
                    for reason, field, is_minimum in [("minimum duration", "trip_duration_minutes", True),
                                                       ("maximum duration", "trip_duration_minutes", False),
                                                       ("maximum speed", "speed_mph", False)]:
                        if a_enriched[field].notna().any():
                            index = a_enriched[field].idxmin() if is_minimum else a_enriched[field].idxmax()
                            value = a_enriched.at[index, field]
                            if (value < a_extrema[reason]) if is_minimum else (value > a_extrema[reason]):
                                a_extrema[reason] = value
                                a_samples[reason] = a_sample_record(a_enriched, index, a_path.name, reason)
                    if "fallback" not in a_samples and len(a_enriched) > 1:
                        a_samples["fallback"] = a_sample_record(a_enriched, a_enriched.index[1], a_path.name, "additional source example")
                    del a_enriched, a_chunk, valid_hours
            a_file_rows.append({"source_file": a_path.name, "rows": a_this_file_rows})
            print(f"Finished {a_path.name}: {a_this_file_rows:,} rows; audit fields calculated without row filtering", flush=True)
    finally:
        for handle in a_handles.values():
            handle.close()
    a_summary_records = []
    for field, stats in a_stats.items():
        print(f"Calculating exact disk-backed median: {field}", flush=True)
        a_summary_records.append({"field": field,
            "min": stats["min"] if stats["n"] else np.nan,
            "median": a_exact_median(a_temp_paths[field], stats["n"]),
            "mean": math.fsum(stats["chunk_sums"]) / stats["n"] if stats["n"] else np.nan,
            "max": stats["max"] if stats["n"] else np.nan})
a_summary = pd.DataFrame(a_summary_records).set_index("field")
assert not Path(a_tmp).exists()
print("PASS: temporary median files removed.")
print("Rows processed:", a_rows)
print(pd.DataFrame(a_file_rows).to_string(index=False))


Finished Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 3,574,091 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 4,251,015 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 4,428,699 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 4,181,444 rows; audit fields calculated without row filtering
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 

### Ten-row calculation validation

Display original timestamps, distance, location IDs, all eight calculated fields, source filename, and source row number (one-based data row, excluding the header). Independently recompute each displayed result from the scalar source values and check it against the chunk calculations. Speed must remain missing when duration is zero, negative, or undefined. No sample row is used to prescribe a cleaning decision.

In [17]:
a_sample_keys = [f"file {index}" for index in sorted(a_sample_file_indices)]
a_sample_keys += ["zero duration" if "zero duration" in a_samples else "fallback", "minimum duration", "maximum duration", "maximum speed"]
a_sample = pd.DataFrame([a_samples[key] for key in a_sample_keys]).reset_index(drop=True)
assert len(a_sample) == 10
for _, row in a_sample.iterrows():
    pickup = pd.to_datetime(row[a_columns["pickup"]], errors="coerce", format="mixed")
    dropoff = pd.to_datetime(row[a_columns["dropoff"]], errors="coerce", format="mixed")
    expected_duration = (dropoff - pickup).total_seconds() / 60 if pd.notna(pickup) and pd.notna(dropoff) else np.nan
    assert np.isclose(row["trip_duration_minutes"], expected_duration, equal_nan=True)
    if expected_duration > 0 and pd.notna(row[a_columns["distance"]]):
        expected_speed = row[a_columns["distance"]] / (expected_duration / 60)
        assert np.isclose(row["speed_mph"], expected_speed, equal_nan=True)
    else:
        assert pd.isna(row["speed_mph"])
    if pd.notna(pickup):
        assert row["pickup_date"] == pickup.date()
        assert row["pickup_hour"] == pickup.hour
        assert row["day_of_week"] == pickup.day_name()
        assert row["month"] == pickup.month
        assert row["weekend"] == (pickup.weekday() >= 5)
    else:
        assert all(pd.isna(row[field]) for field in ["pickup_date", "pickup_hour", "day_of_week", "month", "weekend"])
    if pd.notna(row[a_columns["origin"]]) and pd.notna(row[a_columns["destination"]]):
        assert row["route_id"] == f"{row[a_columns['origin']]}-{row[a_columns['destination']]}"
    else:
        assert pd.isna(row["route_id"])
print(a_sample.to_string(index=False))
print("PASS: all ten sample rows match independent scalar calculations.")


   pickup_timestamp   dropoff_timestamp  distance_miles  origin_loc_id  dest_loc_id  trip_duration_minutes    speed_mph pickup_date  pickup_hour day_of_week  month  weekend route_id                                   source_file  source_row_1based      sample_reason
2025-04-01 00:47:06 2025-04-01 01:13:25            9.50            138          230              26.316667 2.165928e+01  2025-04-01            0     Tuesday      4    False  138-230 Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv                  1 file-start example
2025-06-01 00:02:50 2025-06-01 00:39:51           10.00            138           50              37.016667 1.620891e+01  2025-06-01            0      Sunday      6     True   138-50 Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv                  1 file-start example
2025-08-01 00:52:23 2025-08-01 01:12:20            8.44            138          141              19.950000 2.538346e+01  2025-08-01            0      Friday      8    False  138-141 Urban_Flow_Analytics

### Descriptive summaries and calculation checks

Report min, exact median, mean, and max of defined duration/speed values. Non-positive durations are included in duration summaries and have undefined speed. Undefined values are excluded only from the relevant numeric summary, not from the data. Weekday and weekend totals include a missing category if needed, so every source row is accounted for. No anomaly counts or cleaning recommendations are calculated.

In [18]:
print("New audit fields and observed dtypes:", {field: sorted(types) for field, types in a_field_dtypes.items()})
print("Source-column dtypes:", {column: sorted(types) for column, types in a_dtypes.items()})
print("Duration (minutes) and speed (mph) summaries:")
print(a_summary.to_string(float_format=lambda value: f"{value:.9f}"))
print("Pickup-hour range:", (a_hour_min, a_hour_max))
print("Weekday counts:")
for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
    print(f"{day}: {a_weekdays.get(day, 0):,}")
for day, count in a_weekdays.items():
    if day not in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
        print(f"Missing/other weekday ({day}): {count:,}")
print("Weekend versus weekday counts:", dict(a_weekends))
assert sum(a_weekdays.values()) == a_rows
assert sum(a_weekends.values()) == a_rows
assert a_hour_min is None or 0 <= a_hour_min <= a_hour_max <= 23
assert a_snapshot == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in a_files}
print("PASS: calendar counts reconcile to all source rows; source sizes and modification times unchanged.")
print("STOP: Step 4 calculations only. No cleaning, anomaly counts, joined/enriched dataset output, or later processing.")


New audit fields and observed dtypes: {'trip_duration_minutes': ['float64'], 'speed_mph': ['float64'], 'pickup_date': ['object'], 'pickup_hour': ['Int64'], 'day_of_week': ['str'], 'month': ['Int64'], 'weekend': ['boolean'], 'route_id': ['string']}
Source-column dtypes: {'pickup_timestamp': ['str'], 'dropoff_timestamp': ['str'], 'distance_miles': ['float64'], 'origin_loc_id': ['int64'], 'dest_loc_id': ['int64']}
Duration (minutes) and speed (mph) summaries:
                                 min       median         mean               max
field                                                                           
trip_duration_minutes -707.316666667 13.850000000 17.799116685   14880.766666667
speed_mph                0.000000000  9.230769231 24.309371532 4816614.800000000
Pickup-hour range: (0, 23)
Weekday counts:
Monday: 5,894,446
Tuesday: 6,893,719
Wednesday: 7,194,184
Thursday: 7,494,630
Friday: 7,283,768
Saturday: 7,484,201
Sunday: 6,356,834
Weekend versus weekday counts: {'weekd

### Step 4 recorded findings

All eight requested fields were created in temporary chunk audit views: `trip_duration_minutes`, `speed_mph`, `pickup_date`, `pickup_hour`, `day_of_week`, `month`, `weekend`, and `route_id`. All **48,601,782 rows across 12 files** were processed without concatenating the full data or removing rows. The source fields had consistent types in every chunk: timestamps were strings, distance was `float64`, and origin/destination IDs were `int64`. No essential columns were missing and no column/type issues were encountered.

| Descriptive statistic | Trip duration (minutes) | Speed (mph; defined on positive-duration rows) |
|---|---:|---:|
| Minimum | -707.316666667 | 0.000000000 |
| Exact median | 13.850000000 | 9.230769231 |
| Mean | 17.799116685 | 24.309371532 |
| Maximum | 14,880.766666667 | 4,816,614.800000000 |

The medians are exact over the defined numeric calculation values, using temporary disk-backed arrays. All temporary files were removed; no enriched/cleaned dataset or `clean_trips.parquet` was saved.

- **Pickup-hour range:** 0-23.
- **Weekdays:** 34,760,747 rows. **Weekends:** 13,841,035 rows. These sum to all 48,601,782 source rows; the full weekday breakdown appears above.
- **Validation:** all ten displayed sample rows passed independent scalar checks for duration, speed, calendar fields, and route ID. Each chunk also confirmed missing speed wherever duration was non-positive or undefined.
- **Unexpected recorded results:** negative and very long durations, and an extreme maximum speed, remain visible in the descriptive summaries. The maximum-speed example uses a source distance of **240,830.74 miles** over **3 minutes**, producing **4,816,614.8 mph** under the requested formula. The zero-duration and negative-duration sample rows correctly have missing speed. These observations are not anomaly counts, thresholds, or cleaning decisions.
- **Preservation:** Steps 1-3 cells and outputs were preserved; source sizes and modification times were unchanged. No source values were imputed or modified.

**Stop after Step 4.** No anomaly counting, cleaning rules, feature contract, splitting, or machine learning was performed.

## Step 5: Anomaly measurement and inspection

Measure anomalies before making cleaning decisions. This section runs independently, preserves Steps 1-4, and processes all monthly CSVs in 200,000-row chunks. No source rows are removed, overwritten, or imputed. Calculated duration and speed are temporary audit fields; speed is defined only for positive duration.

All impact percentages use **48,601,782 source rows**. Checks overlap, and speed thresholds are nested: counts must not be added to estimate distinct affected rows. **80, 100, and 120 mph are inspection thresholds only**, not accepted cleaning cutoffs. Historical-year and out-of-month checks use both timestamps separately and also report an either-timestamp union to avoid double-counting within that union.

In [19]:
from pathlib import Path
import re
import tempfile
import numpy as np
import pandas as pd

m_root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"data"/"raw").is_dir())
m_files=sorted(p for p in (m_root/"data"/"raw").rglob("Urban_Flow_Analytics_Taxi_Dataset_*.csv") if p.is_file()
               and "__MACOSX" not in p.parts and not any(part.startswith("._") for part in p.parts))
assert len(m_files)==12
m_expected_rows=48_601_782
m_required=["pickup_timestamp","dropoff_timestamp","origin_loc_id","dest_loc_id","distance_miles","base_fare","rider_count"]
m_optional=["fare_settlement_method","provider_code","rate_class_id"]
m_usecols={}
for p in m_files:
    header=pd.read_csv(p,nrows=0).columns.tolist()
    missing=set(m_required)-set(header)
    if missing:
        raise ValueError(f"Missing essential columns in {p.name}: {sorted(missing)}")
    m_usecols[p]=m_required+[column for column in m_optional if column in header]
m_snapshot={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in m_files}
m_notes={
    "Negative base_fare":"Observed signed fare; refund/reversal semantics require documentation.",
    "Negative distance_miles":"Negative physical trip distance; measure without deciding treatment.",
    "Zero distance with nonzero base_fare":"Requires observed non-missing fare; not automatically a trip error.",
    "rider_count == 0":"Zero recorded riders; semantics/treatment not decided.",
    "Duration <= 0":"Includes negative and zero duration; speed remains missing.",
    "Duration < 0":"Dropoff precedes pickup as recorded; subset of duration <= 0.",
    "Duration == 0":"Identical parsed timestamps; subset of duration <= 0.",
    "Speed > 80 mph":"Inspection only; defined speed on positive-duration rows.",
    "Speed > 100 mph":"Inspection only; nested within >80 mph.",
    "Speed > 120 mph":"Inspection only; nested within >100 mph.",
    "Pickup year 2008/2009":"Calendar year from parsed pickup; retain rows.",
    "Dropoff year 2008/2009":"Calendar year from parsed dropoff; retain rows.",
    "Either timestamp year 2008/2009":"Row union of historical pickup/dropoff checks.",
    "Pickup outside filename month":"Valid pickup before month start or on/after next month.",
    "Dropoff outside filename month":"May include legitimate month-boundary trips.",
    "Either timestamp outside filename month":"Row union of pickup/dropoff month mismatches.",
}
print("Verified columns:",m_required)
print("Optional inspection codes:",m_optional)
print("Exact source files:")
for p in m_files: print(p.name)


Verified columns: ['pickup_timestamp', 'dropoff_timestamp', 'origin_loc_id', 'dest_loc_id', 'distance_miles', 'base_fare', 'rider_count']
Optional inspection codes: ['fare_settlement_method', 'provider_code', 'rate_class_id']
Exact source files:
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


### Incremental measurements and representative examples

Recompute duration and speed with the Step 4 formulas. Missing inputs do not satisfy numerical comparisons; specifically, a missing fare is not treated as a nonzero fare. Parse timestamps with `errors="coerce"`, `format="mixed"`, without assigning a timezone. Filename-month checks compare valid timestamps to the calendar month in each source name.

For each check, retain up to **10 reproducible random-priority inspection examples** across all matching rows, with a fixed seed per check. These small samples are not proof of a cause. If fewer than 10 records exist, show every available record; if none exist, state that explicitly. Each example includes filename/month, a one-based source data-row number, original values, duration/speed, and available payment/provider/rate codes.

For exact speed percentiles, append only defined positive-duration speeds to one temporary numeric file, partition a disk-backed array at the required order statistics, and interpolate using NumPy's default linear quantile convention. The temporary file is removed afterward. No enriched or cleaned dataset is saved.

In [20]:
m_counts={name:0 for name in m_notes}
m_samples={name:pd.DataFrame() for name in m_notes}
m_rng={name:np.random.default_rng(20260500+i) for i,name in enumerate(m_notes)}
m_file_results=[]
m_rows=0
m_speed_n=0
m_parse_failures={"pickup":0,"dropoff":0}
m_source_dtypes={column:set() for column in m_required}
with tempfile.TemporaryDirectory(prefix="urban_flow_step5_speed_") as m_tmp:
    m_speed_path=Path(m_tmp)/"speed.bin"
    with m_speed_path.open("wb") as m_speed_output:
        for m_path in m_files:
            m_month=re.search(r"(\d{4}-\d{2})\.csv$",m_path.name).group(1)
            m_start=pd.Timestamp(m_month+"-01")
            m_end=m_start+pd.offsets.MonthBegin(1)
            m_result={"source_file":m_path.name,"source_month":m_month,"rows":0,**{name:0 for name in m_notes}}
            with pd.read_csv(m_path,usecols=m_usecols[m_path],parse_dates=False,chunksize=200_000) as m_reader:
                for frame in m_reader:
                    for column in m_required: m_source_dtypes[column].add(str(frame[column].dtype))
                    for column in ["base_fare","distance_miles","rider_count"]:
                        if not pd.api.types.is_numeric_dtype(frame[column]):
                            raise TypeError(f"Unexpected nonnumeric {column} in {m_path.name}; no coercion applied")
                    pickup=pd.to_datetime(frame["pickup_timestamp"],errors="coerce",format="mixed",utc=False)
                    dropoff=pd.to_datetime(frame["dropoff_timestamp"],errors="coerce",format="mixed",utc=False)
                    if pickup.dt.tz is not None or dropoff.dt.tz is not None:
                        raise TypeError("Timezone representation changed from earlier steps; review required")
                    m_parse_failures["pickup"]+=int(pickup.isna().sum())
                    m_parse_failures["dropoff"]+=int(dropoff.isna().sum())
                    frame["trip_duration_minutes"]=(dropoff-pickup).dt.total_seconds()/60
                    positive=frame["trip_duration_minutes"]>0
                    frame["speed_mph"]=np.nan
                    frame.loc[positive,"speed_mph"]=frame.loc[positive,"distance_miles"]/(frame.loc[positive,"trip_duration_minutes"]/60)
                    assert frame.loc[~positive,"speed_mph"].isna().all()
                    speeds=frame["speed_mph"].dropna().to_numpy(dtype="float64")
                    speeds.tofile(m_speed_output)
                    m_speed_n+=len(speeds)
                    del speeds
                    old_pickup=pickup.dt.year.isin([2008,2009])
                    old_dropoff=dropoff.dt.year.isin([2008,2009])
                    outside_pickup=(pickup<m_start)|(pickup>=m_end)
                    outside_dropoff=(dropoff<m_start)|(dropoff>=m_end)
                    masks={
                        "Negative base_fare":frame["base_fare"]<0,
                        "Negative distance_miles":frame["distance_miles"]<0,
                        "Zero distance with nonzero base_fare":(frame["distance_miles"]==0)&frame["base_fare"].notna()&(frame["base_fare"]!=0),
                        "rider_count == 0":frame["rider_count"]==0,
                        "Duration <= 0":frame["trip_duration_minutes"]<=0,
                        "Duration < 0":frame["trip_duration_minutes"]<0,
                        "Duration == 0":frame["trip_duration_minutes"]==0,
                        "Speed > 80 mph":frame["speed_mph"]>80,
                        "Speed > 100 mph":frame["speed_mph"]>100,
                        "Speed > 120 mph":frame["speed_mph"]>120,
                        "Pickup year 2008/2009":old_pickup,
                        "Dropoff year 2008/2009":old_dropoff,
                        "Either timestamp year 2008/2009":old_pickup|old_dropoff,
                        "Pickup outside filename month":outside_pickup,
                        "Dropoff outside filename month":outside_dropoff,
                        "Either timestamp outside filename month":outside_pickup|outside_dropoff,
                    }
                    for name,mask in masks.items():
                        positions=np.flatnonzero(mask.to_numpy(dtype=bool))
                        count=len(positions)
                        m_counts[name]+=count
                        m_result[name]+=count
                        if count:
                            priorities=m_rng[name].random(count)
                            chosen=np.argpartition(priorities, min(10,count)-1)[:10]
                            sample=frame.iloc[positions[chosen]].copy()
                            sample["source_file"]=m_path.name
                            sample["source_month"]=m_month
                            sample["source_row_1based"]=sample.index+1
                            sample["_priority"]=priorities[chosen]
                            previous=m_samples[name]
                            candidates=pd.concat([previous,sample],ignore_index=True) if not previous.empty else sample
                            m_samples[name]=candidates.nsmallest(10,"_priority").copy()
                    m_rows+=len(frame)
                    m_result["rows"]+=len(frame)
                    del frame,pickup,dropoff,masks
            m_file_results.append(m_result)
            print(f"Finished {m_path.name}: {m_result['rows']:,} rows inspected",flush=True)
    print("Computing exact speed percentiles",flush=True)
    m_quantiles=np.array([0.5,0.9,0.95,0.99,0.995,0.999,1.0])
    if not m_speed_n: raise ValueError("No defined positive-duration speeds to summarize")
    vector=np.memmap(m_speed_path,dtype="float64",mode="r+",shape=(m_speed_n,))
    try:
        ranks=(m_speed_n-1)*m_quantiles
        low=np.floor(ranks).astype("int64")
        high=np.ceil(ranks).astype("int64")
        vector.partition(np.unique(np.concatenate([low,high])))
        values=np.array([float(vector[l]) if l==h else float(vector[l])+(float(vector[h])-float(vector[l]))*(r-l)
                         for l,h,r in zip(low,high,ranks)])
        m_speed_percentiles=pd.DataFrame({"Percentile":["50th","90th","95th","99th","99.5th","99.9th","Maximum"],"Speed mph":values})
        vector.flush()
    finally:
        vector._mmap.close()
assert not Path(m_tmp).exists()
assert m_rows==m_expected_rows
m_by_file=pd.DataFrame(m_file_results)
for name in m_notes:
    assert int(m_by_file[name].sum())==m_counts[name]
    assert len(m_samples[name])==min(10,m_counts[name])
print("PASS: all source rows accounted for; per-file counts reconcile; temporary speed file removed.")


Finished Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 3,574,091 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 4,251,015 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 4,428,699 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 4,181,444 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 4,305,006 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv: 3,724,889 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv: 3,399,866 rows inspected
Finished Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv: 3,952,451 rows inspected
Computing exact speed percen

### Speed distribution before threshold decisions

These exact percentiles describe defined speed values on positive-duration rows. Zero distances and extreme speeds remain included. Undefined speeds are excluded only from this distribution, never by removing source rows. Example threshold impacts follow in the table; no final speed threshold is selected.

In [21]:
print("Rows with defined positive-duration speed:",m_speed_n)
print(m_speed_percentiles.to_string(index=False,float_format=lambda x:f"{x:,.9f}"))


Rows with defined positive-duration speed: 47950172
Percentile           Speed mph
      50th         9.230769231
      90th        19.007462687
      95th        24.046547711
      99th        34.258064516
    99.5th        38.103382022
    99.9th        46.850574713
   Maximum 4,816,614.800000000


### Anomaly impact and monthly source counts

The impact table uses all 48,601,782 rows as denominator for every check. Counts overlap; historical and month-mismatch union rows are explicitly identified. Monthly counts refer to the source CSV, not the derived pickup month, so suspicious timestamps do not move a row between source-file groups.

In [22]:
m_impact=pd.DataFrame([{"Anomaly":name,"Count":m_counts[name],"Percent of Total":m_counts[name]/m_expected_rows*100,"Notes":note}
                       for name,note in m_notes.items()])
print(m_impact.to_string(index=False,float_format=lambda x:f"{x:.9f}"))
print("Source file row counts:")
print(m_by_file[["source_file","rows"]].to_string(index=False))
print("Anomaly counts by source month (full filenames listed above):")
print(m_by_file.set_index("source_month")[list(m_notes)].T.to_string())
print("Timestamp parsing failures (not assigned a year/month):",m_parse_failures)
print("Observed source dtypes:",{column:sorted(types) for column,types in m_source_dtypes.items()})


                                Anomaly   Count  Percent of Total                                                                  Notes
                     Negative base_fare 2400031       4.938154325 Observed signed fare; refund/reversal semantics require documentation.
                Negative distance_miles       0       0.000000000   Negative physical trip distance; measure without deciding treatment.
   Zero distance with nonzero base_fare 1471746       3.028172918    Requires observed non-missing fare; not automatically a trip error.
                       rider_count == 0  231578       0.476480471                 Zero recorded riders; semantics/treatment not decided.
                          Duration <= 0  651610       1.340712157            Includes negative and zero duration; speed remains missing.
                           Duration < 0    1942       0.003995738          Dropoff precedes pickup as recorded; subset of duration <= 0.
                          Duration == 0  

### Representative anomaly records

Show up to ten seeded inspection examples per check, ordered by source file and row number for readability. Overlapping checks can show the same row. Fewer than five examples is possible when fewer than five records exist; zero-count checks have no examples to show. Source values are displayed unchanged, alongside temporary duration and speed calculations.

In [23]:
for name in m_notes:
    print("\nANOMALY:",name,"| total rows:",m_counts[name])
    if m_samples[name].empty:
        print("No matching records; no examples available.")
    else:
        shown=m_samples[name].drop(columns="_priority").sort_values(["source_file","source_row_1based"])
        leading=["source_file","source_month","source_row_1based"]
        print(shown[leading+[column for column in shown.columns if column not in leading]].to_string(index=False))
assert m_snapshot=={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in m_files}
print("PASS: source sizes and modification times unchanged.")
print("STOP: Step 5 measurement and inspection only. No rows cleaned, removed, imputed, or saved as cleaned data.")



ANOMALY: Negative base_fare | total rows: 2400031
                                  source_file source_month  source_row_1based  provider_code    pickup_timestamp   dropoff_timestamp  rider_count  distance_miles  rate_class_id  origin_loc_id  dest_loc_id  fare_settlement_method  base_fare  trip_duration_minutes  speed_mph
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv      2025-04            3646483              2 2025-04-18 03:41:11 2025-04-18 03:47:12          NaN            0.70            NaN            234          164                       0      -4.75               6.016667   6.980609
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv      2025-05            4310201              2 2025-05-23 23:37:33 2025-05-23 23:45:46          NaN            1.32            NaN            211          231                       0      -3.22               8.216667   9.638945
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv      2025-06              32963              2 2025-06-01 12:02:54 2025-06-01 12:

### Step 5 recorded findings and interpretation

All **48,601,782 rows across 12 files** were measured without cleaning or filtering. The impact table above contains exact counts, percentages, notes, and monthly source counts. Every nonempty check includes 10 reproducible inspection examples, except the historical-date checks, which show **all 8 records**. Negative distance has no records to show. Counts overlap and must not be summed.

| Check | Count | Percent of all rows |
|---|---:|---:|
| Negative base fare | 2,400,031 | 4.938154325% |
| Negative distance | 0 | 0% |
| Zero distance with nonzero fare | 1,471,746 | 3.028172918% |
| Zero rider count | 231,578 | 0.476480471% |
| Non-positive duration | 651,610 | 1.340712157% |
| Negative duration (subset) | 1,942 | 0.003995738% |
| Zero duration (subset) | 649,668 | 1.336716419% |
| Speed above 80 mph (inspection only) | 13,589 | 0.027959880% |
| Speed above 100 mph (inspection only) | 11,899 | 0.024482641% |
| Speed above 120 mph (inspection only) | 10,672 | 0.021958043% |

**Exact speed distribution**, using 47,950,172 defined speeds on positive-duration records: p50 **9.230769231**, p90 **19.007462687**, p95 **24.046547711**, p99 **34.258064516**, p99.5 **38.103382022**, p99.9 **46.850574713**, maximum **4,816,614.8 mph**. All defined speeds were retained, including zero and extreme values; no threshold was selected. The temporary numeric file was removed.

**Suspicious timestamps:** pickup year 2008/2009: **8**; dropoff year 2008/2009: **8**; either-timestamp union: **8 (0.000016460%)**. These are the same eight rows: May 2025 **1**, July **1**, August **1**, November **3**, and March 2026 **2**. Out-of-month pickups: **170 (0.000349781%)**; dropoffs: **19,538 (0.040200172%)**; either-timestamp union: **19,646 (0.040422386%)**. There were no timestamp parsing failures.

#### Values that appear inconsistent with the intended measurements

- The extreme implied speeds, including **4,816,614.8 mph**, cannot represent taxi road travel. At least one contributing value or its interpretation is wrong; this does not identify which source field to correct or establish a universal speed cutoff.
- Dates in **2008/2009** are incompatible with the stated 2025-2026 observation period. The audit cannot infer replacement dates or whether these represent intentionally included historical records.
- Negative calculated durations are unusable as elapsed travel times **as currently calculated**. Zero duration with positive distance is also inconsistent with a literal movement-time interpretation. This is not evidence that every underlying trip should be deleted.

#### Ambiguous cases needing documentation or a later policy decision

- **Negative fares:** possible financial adjustments, reversals, or source errors; no payment/rate code meanings are assumed. Source-month counts change substantially, including 395,093 in November and 48,175 in December; causes are untested.
- **Zero distance with nonzero fare:** samples include same and different origin/destination IDs and different rate/payment codes. The source does not establish whether these are special charges, incomplete measurements, or errors.
- **Zero riders:** possible recording conventions or data problems; zero is not imputed to a passenger count.
- **Non-positive duration:** samples include zero timestamps with positive distance. Several negative-duration examples occur on November 2 within the 01:00 hour; a backward-clock/timezone ambiguity is one possible explanation, **not verified**. The source is timezone-naive, so the audit does not assign a timezone or change timestamps. November contains 1,435 of the 1,942 negative-duration rows; no cause is inferred from that concentration.
- **Speeds above inspection thresholds:** examples range from just above a threshold to extreme values. Neither a percentile nor an example threshold determines final acceptability.
- **Month mismatches:** examples include ordinary trips crossing midnight at month-end. Source-file membership alone is not proof of an invalid trip; historical dates and other mismatches require separate interpretation.

Steps 1-4 remain unchanged. Monthly counts reconcile to the full row count, source sizes and modification times remain unchanged, and no source values were overwritten or imputed. **Stop after Step 5:** no final treatment, cleaning rules, cleaned Parquet output, splits, or modelling.

## Step 6: Investigation and proposed cleaning policy

Investigation and policy design only: no source values or rows are changed, and Steps 1-5 remain unchanged. All 12 files are processed sequentially in 200,000-row chunks. Subgroup selections below are temporary measurements, not cleaning. Counts, means, minima/maxima, missingness, code distributions, pairwise overlaps, and proposed-rule unions are exact. Subgroup p50/p95 values use reproducible random-priority samples of at most 5,000 records per subgroup and are explicitly approximate; the Step 5 full-data speed percentiles remain the exact reference.

Groups overlap. Compare zero riders with **positive recorded riders**, not an assumed clean population. Compare negative fares with nonnegative fares, and zero-distance charged rows with positive-distance rows. Missing riders are not silently classified as normal.

External context: the [TLC dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) distinguishes payment and rate types, but the competition's renamed columns are not documented as exact copies. Code meanings are therefore conditional, not assigned facts. [TLC fare rules](https://www.nyc.gov/site/tlc/passengers/taxi-fare.page) include time-based charging during slow movement or waiting, so a zero distance alone cannot invalidate a fare. [NIST clock-change rules](https://www.nist.gov/pml/time-and-frequency-division/popular-links/daylight-saving-time-dst) motivate inspection of the repeated November hour; no timezone is assigned or timestamps corrected here.

In [24]:
from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd
s_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"data"/"raw").is_dir())
s_files=sorted(p for p in (s_root/"data"/"raw").rglob("Urban_Flow_Analytics_Taxi_Dataset_*.csv") if p.is_file() and "__MACOSX" not in p.parts and not any(x.startswith("._") for x in p.parts))
assert len(s_files)==12
s_snapshot={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in s_files}
s_required=["pickup_timestamp","dropoff_timestamp","distance_miles","base_fare","rider_count","origin_loc_id","dest_loc_id","charge_total","provider_code","fare_settlement_method","rate_class_id","offline_record_flag"]
for p in s_files:
    missing=set(s_required)-set(pd.read_csv(p,nrows=0).columns)
    if missing: raise ValueError(f"Missing fields in {p.name}: {missing}")
s_group_names=["negative_fare","nonnegative_fare","zero_distance_charged","positive_distance","zero_riders","positive_riders","negative_duration","zero_duration","speed_80_to_100","speed_100_to_120","speed_over_120"]
s_metrics=["base_fare","charge_total","distance_miles","trip_duration_minutes","speed_mph"]
s_code_cols=["fare_settlement_method","provider_code","rate_class_id","offline_record_flag"]
s_sizes=Counter(); s_codes={g:{col:Counter() for col in s_code_cols} for g in s_group_names}
s_missing={g:Counter() for g in s_group_names}
s_numeric={g:{col:{"n":0,"sum":0.,"min":float("inf"),"max":-float("inf")} for col in s_metrics} for g in s_group_names}
s_samples={g:pd.DataFrame() for g in s_group_names}
s_rng={g:np.random.default_rng(6000+i) for i,g in enumerate(s_group_names)}
s_rel={g:Counter() for g in s_group_names}
s_baseline_codes={col:Counter() for col in s_code_cols}
s_pair_codes=Counter(); s_hist=[]; s_boundary_samples={}; s_boundary_counts=Counter(); s_rules=Counter(); s_monthly=[]
s_anomaly_names=["negative_fare","negative_distance","zero_distance_charged","zero_riders","nonpositive_duration","speed_over_100","historical","out_of_month"]
s_combinations=np.zeros(256,dtype="int64")
s_rule_names=["historical_quarantine","outside_observation_nonhistorical","negative_fare_or_total_model","negative_distance_model","nonpositive_duration_model","speed_over_100_model"]
s_rule_combinations=np.zeros(64,dtype="int64")
s_rows=0


### Chunked investigation and proposal impact measurement

Inspect signed final charges, other monetary components, codes, missingness, location equality, and duration/distance distributions. Sign associations can support an operational-record hypothesis but cannot prove a refund, reversal, or void without linked transactions or a confirmed dictionary.

Month-mismatch partitions (inspection categories, not deletion rules): historical; nonhistorical records touching the adjacent month boundary with positive duration at most two hours and timestamps within two hours of that boundary; other adjacent-boundary records; remaining mismatches. The two-hour window is only an explanatory bucket; changing it does not change the proposed removal policy. Sample every partition that exists.

The proposed model-eligibility scenario measured here excludes out-of-observation pickups, negative base/final fares, negative distances, non-positive durations, and speeds above 100 mph. It retains zero riders and zero-distance charged trips unless another rule applies. All rules are Boolean measurements only. Historical quarantine and observation scope are separate from task-specific model exclusions; raw archives always retain everything.

In [25]:
for path in s_files:
    month=path.stem[-7:]; start=pd.Timestamp(month+"-01"); end=start+pd.offsets.MonthBegin(1)
    monthly=Counter(); monthly["source_month"]=month
    with pd.read_csv(path,chunksize=200_000,parse_dates=False) as reader:
        for f in reader:
            pickup=pd.to_datetime(f.pickup_timestamp,errors="coerce",format="mixed")
            dropoff=pd.to_datetime(f.dropoff_timestamp,errors="coerce",format="mixed")
            assert pickup.notna().all() and dropoff.notna().all() and pickup.dt.tz is None and dropoff.dt.tz is None
            f["trip_duration_minutes"]=(dropoff-pickup).dt.total_seconds()/60
            positive=f.trip_duration_minutes>0
            f["speed_mph"]=np.nan
            f.loc[positive,"speed_mph"]=f.loc[positive,"distance_miles"]/(f.loc[positive,"trip_duration_minutes"]/60)
            groups={"negative_fare":f.base_fare<0,"nonnegative_fare":f.base_fare>=0,
                "zero_distance_charged":f.distance_miles.eq(0)&f.base_fare.notna()&f.base_fare.ne(0),"positive_distance":f.distance_miles>0,
                "zero_riders":f.rider_count.eq(0),"positive_riders":f.rider_count>0,
                "negative_duration":f.trip_duration_minutes<0,"zero_duration":f.trip_duration_minutes.eq(0),
                "speed_80_to_100":(f.speed_mph>80)&(f.speed_mph<=100),"speed_100_to_120":(f.speed_mph>100)&(f.speed_mph<=120),"speed_over_120":f.speed_mph>120}
            for col in s_code_cols:
                s_baseline_codes[col].update({str(k):int(v) for k,v in f[col].value_counts(dropna=False).items()})
            for g,mask in groups.items():
                selected=f.loc[mask]; count=len(selected); s_sizes[g]+=count
                if not count: continue
                s_missing[g].update({k:int(v) for k,v in selected.isna().sum().items()})
                for col in s_code_cols: s_codes[g][col].update({str(k):int(v) for k,v in selected[col].value_counts(dropna=False).items()})
                for col in s_metrics:
                    vals=selected[col].dropna(); stat=s_numeric[g][col]
                    if len(vals):
                        stat["n"]+=len(vals); stat["sum"]+=float(vals.sum()); stat["min"]=min(stat["min"],float(vals.min())); stat["max"]=max(stat["max"],float(vals.max()))
                relationships={"same_origin_destination":selected.origin_loc_id.eq(selected.dest_loc_id),"positive_duration":selected.trip_duration_minutes>0,
                    "negative_final_charge":selected.charge_total<0,"zero_final_charge":selected.charge_total.eq(0),"positive_final_charge":selected.charge_total>0,
                    "positive_distance":selected.distance_miles>0,"duration_at_most_one_minute":selected.trip_duration_minutes.between(0,1,inclusive="right")}
                money_cols=[col for col in ["surcharge_misc","transit_tax","driver_tip_payment","toll_total","service_improvement_fee","zone_congestion_fee","Airport_fee","congestion_relief_fee"] if col in selected]
                relationships["any_negative_other_monetary_component"]=(selected[money_cols]<0).any(axis=1)
                for name,m in relationships.items(): s_rel[g][name]+=int(m.sum())
                priorities=s_rng[g].random(count); chosen=np.argpartition(priorities,min(5000,count)-1)[:5000]
                sample=selected.iloc[chosen].copy(); sample["source_file"]=path.name; sample["source_row_1based"]=sample.index+1; sample["_priority"]=priorities[chosen]
                candidates=pd.concat([s_samples[g],sample],ignore_index=True) if not s_samples[g].empty else sample
                s_samples[g]=candidates.nsmallest(5000,"_priority").copy()
                if g=="negative_fare":
                    s_pair_codes.update({str(k):int(v) for k,v in selected.groupby(["fare_settlement_method","provider_code"],dropna=False).size().items()})
            historical=pickup.dt.year.isin([2008,2009])|dropoff.dt.year.isin([2008,2009])
            outmonth=(pickup<start)|(pickup>=end)|(dropoff<start)|(dropoff>=end)
            if historical.any():
                h=f.loc[historical].copy(); h["source_file"]=path.name; h["source_row_1based"]=h.index+1; s_hist.append(h)
            near_boundary=pd.Series(False,index=f.index)
            for b in [start,end]:
                near_boundary|=(pickup-b).abs().le(pd.Timedelta(hours=2))&(dropoff-b).abs().le(pd.Timedelta(hours=2))
            short_boundary=outmonth&~historical&near_boundary&positive&f.trip_duration_minutes.le(120)
            adjacent_any=((pickup>=start-pd.Timedelta(days=1))&(pickup<end+pd.Timedelta(days=1))&(dropoff>=start-pd.Timedelta(days=1))&(dropoff<end+pd.Timedelta(days=1)))
            other_adjacent=outmonth&~historical&~short_boundary&adjacent_any
            remaining=outmonth&~historical&~short_boundary&~other_adjacent
            partitions={"historical":historical,"short_near_boundary":short_boundary,"other_adjacent_boundary":other_adjacent,"remaining_month_mismatch":remaining}
            for name,mask in partitions.items():
                s_boundary_counts[name]+=int(mask.sum())
                prior=s_boundary_samples.get(name,pd.DataFrame())
                if len(prior)<10 and mask.any():
                    sample=f.loc[mask].head(10-len(prior)).copy(); sample["source_file"]=path.name;sample["source_row_1based"]=sample.index+1
                    s_boundary_samples[name]=pd.concat([prior,sample],ignore_index=True) if len(prior) else sample
            dst=(f.trip_duration_minutes<0)&pickup.dt.normalize().eq(pd.Timestamp("2025-11-02"))&dropoff.dt.normalize().eq(pd.Timestamp("2025-11-02"))&pickup.dt.hour.eq(1)&dropoff.dt.hour.eq(1)&f.trip_duration_minutes.gt(-60)
            s_rules["negative_duration_repeated_hour_candidate"]+=int(dst.sum())
            outperiod=(pickup<pd.Timestamp("2025-04-01"))|(pickup>=pd.Timestamp("2026-04-01"))
            anomaly_masks=[groups["negative_fare"],f.distance_miles<0,groups["zero_distance_charged"],groups["zero_riders"],f.trip_duration_minutes<=0,f.speed_mph>100,historical,outmonth]
            bits=np.zeros(len(f),dtype="uint16")
            for i,mask in enumerate(anomaly_masks):bits|=mask.to_numpy(dtype="uint16")<<i
            s_combinations+=np.bincount(bits,minlength=256)
            rule_masks=[historical,outperiod&~historical,(f.base_fare<0)|(f.charge_total<0),f.distance_miles<0,f.trip_duration_minutes<=0,f.speed_mph>100]
            rbits=np.zeros(len(f),dtype="uint16")
            for i,(name,mask) in enumerate(zip(s_rule_names,rule_masks)):
                s_rules[name]+=int(mask.sum());monthly[name]+=int(mask.sum());rbits|=mask.to_numpy(dtype="uint16")<<i
            s_rule_combinations+=np.bincount(rbits,minlength=64)
            s_rules["zero_riders_in_retained_model_rows"]+=int((groups["zero_riders"]&(rbits==0)).sum())
            s_rules["zero_distance_charged_in_retained_model_rows"]+=int((groups["zero_distance_charged"]&(rbits==0)).sum())
            monthly["model_union"]+=int((rbits!=0).sum()); monthly["rows"]+=len(f);s_rows+=len(f)
            del f,selected,candidates,sample
    s_monthly.append(dict(monthly));print(f"Finished {path.name}: {monthly['rows']:,} rows investigated",flush=True)
assert s_rows==48_601_782 and s_combinations.sum()==s_rows and s_rule_combinations.sum()==s_rows
assert sum(s_boundary_counts.values())==19_646
print("PASS: counts reconcile; proposed rules measured only.")


step6_25:5: DtypeWarning: Columns (0: offline_record_flag) have mixed types. Specify dtype option on import or set low_memory=False.
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows investigated
step6_25:5: DtypeWarning: Columns (0: offline_record_flag) have mixed types. Specify dtype option on import or set low_memory=False.
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows investigated
step6_25:5: DtypeWarning: Columns (0: offline_record_flag) have mixed types. Specify dtype option on import or set low_memory=False.
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows investigated
step6_25:5: DtypeWarning: Columns (0: offline_record_flag) have mixed types. Specify dtype option on import or set low_memory=False.
Finished Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows investigated
step6_25:5: DtypeWarning: Columns (0: offline_record_flag) have mixed types. Specify dtype option on import or set low_memory=False.

### Distributions, code relationships, and missingness

Code tables show subgroup counts, percent within subgroup, and percent of all rows having that code. Code labels stay numeric. Means/minima/maxima and missing counts are exact; sample p50/p95 are approximate and should not be used as precise cutoffs. Positive riders and nonnegative fares are comparison groups, not certified normal records.

In [26]:
s_distribution_rows=[];s_code_rows=[];s_missing_rows=[]
for g in s_group_names:
    for col,stat in s_numeric[g].items():
        sample=s_samples[g][col].dropna() if len(s_samples[g]) else pd.Series(dtype=float)
        s_distribution_rows.append({"group":g,"field":col,"defined_n":stat["n"],"min":stat["min"] if stat["n"] else np.nan,"mean":stat["sum"]/stat["n"] if stat["n"] else np.nan,
          "sample_p50":sample.quantile(.5),"sample_p95":sample.quantile(.95),"max":stat["max"] if stat["n"] else np.nan})
    for col in s_code_cols:
        for value,count in s_codes[g][col].items():
            s_code_rows.append({"group":g,"column":col,"code":value,"count":count,"percent_within_group":100*count/s_sizes[g],"percent_of_all_rows_with_code":100*count/s_baseline_codes[col][value]})
    for col,count in s_missing[g].items():
        s_missing_rows.append({"group":g,"column":col,"missing":count,"percent":100*count/s_sizes[g] if s_sizes[g] else np.nan})
s_distributions=pd.DataFrame(s_distribution_rows);s_code_table=pd.DataFrame(s_code_rows);s_missing_table=pd.DataFrame(s_missing_rows)
print("Exact subgroup sizes:",dict(s_sizes))
print(s_distributions.to_string(index=False))
print("Code relationships:");print(s_code_table.to_string(index=False))
print("Negative fare payment/provider combinations:",dict(s_pair_codes))
print("Missingness comparison, zero vs positive riders:");print(s_missing_table[s_missing_table.group.isin(["zero_riders","positive_riders"])].to_string(index=False))
print("Relationships: count and percent within each subgroup")
for g in s_group_names:
    print(g,{k:{"count":v,"percent":100*v/s_sizes[g]} for k,v in s_rel[g].items()})


Exact subgroup sizes: {'negative_fare': 2400031, 'nonnegative_fare': 46201751, 'zero_distance_charged': 1471746, 'positive_distance': 47123012, 'zero_riders': 231578, 'positive_riders': 35964936, 'negative_duration': 1942, 'zero_duration': 649668, 'speed_80_to_100': 1690, 'speed_100_to_120': 1227, 'speed_over_120': 10672}
                group                 field  defined_n          min         mean  sample_p50    sample_p95           max
        negative_fare             base_fare    2400031 -2555.200000   -10.294165   -4.750000     -1.040000 -1.000000e-02
        negative_fare          charge_total    2400031 -2560.200000    -6.552736    2.485000      8.210500  1.151500e+02
        negative_fare        distance_miles    2400031     0.000000    13.579385    2.300000     11.512000  3.212424e+05
        negative_fare trip_duration_minutes    2400031   -49.283333    18.194396   16.000000     40.166667  4.210117e+03
        negative_fare             speed_mph    2399838     0.000000    

### Inspection records and timestamp partitions

Show ten seeded inspection records per investigated group when available, **all eight historical rows with every original column**, and available boundary-partition examples. A repeated-hour candidate is only a clock-ambiguity hypothesis, not a corrected trip. Month-boundary buckets do not select rows for deletion.

In [27]:
for g in ["negative_fare","zero_distance_charged","zero_riders","negative_duration","zero_duration","speed_80_to_100","speed_100_to_120","speed_over_120"]:
    print("\nInspection group:",g)
    if len(s_samples[g]):print(s_samples[g].head(10).drop(columns="_priority").to_string(index=False))
s_historical=pd.concat(s_hist,ignore_index=True)
assert len(s_historical)==8
print("ALL EIGHT HISTORICAL ROWS, ALL ORIGINAL FIELDS:");print(s_historical.to_string(index=False))
print("Month-mismatch partitions:",dict(s_boundary_counts))
for name,sample in s_boundary_samples.items():print(name);print(sample.to_string(index=False))
print("Negative-duration repeated-hour candidates:",s_rules["negative_duration_repeated_hour_candidate"])



Inspection group: negative_fare
 provider_code    pickup_timestamp   dropoff_timestamp  rider_count  distance_miles  rate_class_id offline_record_flag  origin_loc_id  dest_loc_id  fare_settlement_method  base_fare  surcharge_misc  transit_tax  driver_tip_payment  toll_total  service_improvement_fee  charge_total  zone_congestion_fee  Airport_fee  congestion_relief_fee  trip_duration_minutes  speed_mph                                   source_file  source_row_1based
             2 2025-07-12 21:03:49 2025-07-12 21:07:57          1.0            0.64            1.0                   N             41           42                       4      -6.50            -1.0         -0.5                 0.0         0.0                     -1.0         -9.00                  0.0         0.00                   0.00               4.133333   9.290323 Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv            1016504
             2 2025-10-30 22:45:21 2025-10-30 23:05:06          NaN            1.33        

### Overlap and proposed-rule union

Compute pairwise anomaly overlaps from exact bit-pattern counts. The proposed shared **model-eligibility** union includes all six listed rules; task-specific fare or duration views may choose narrower exclusions after team confirmation. Historical quarantine means exclusion from the analysis view while preserving raw records. Nonhistorical pickups outside the observation interval are scope exclusions, not automatically bad trips. Zero riders and zero-distance charged rows are retained unless another rule excludes them. No numeric values are proposed for imputation.

A **100 mph trip-average speed ceiling** is the conservative proposed model filter, with 80-100 mph retained for review. This is not a legal limit or a claim that every excluded trip is fraudulent. [NY DMV guidance](https://dmv.ny.gov/brochure/mv21.pdf) describes expressway limits normally 55 mph, with some posted at 65 mph. A 100 mph average is about 54% above 65 and more than twice the observed p99.9 of 46.85 mph, providing a substantial allowance for short-record timing/distance error and out-of-area trips. Threshold sensitivity at 80/100/120 and their examples are evaluated before selecting it. This is an explicit, conservative modelling tolerance rather than an estimated physical boundary; team approval remains appropriate.

In [28]:
s_overlap=np.zeros((8,8),dtype="int64")
for bits,count in enumerate(s_combinations):
    for i in range(8):
        for j in range(8):
            if bits&(1<<i) and bits&(1<<j):s_overlap[i,j]+=count
print("Pairwise anomaly overlap; diagonal is each anomaly count:")
print(pd.DataFrame(s_overlap,index=s_anomaly_names,columns=s_anomaly_names).to_string())
s_unique_anomalies=int(s_combinations[1:].sum());s_model_union=int(s_rule_combinations[1:].sum())
s_rule_table=pd.DataFrame([{"rule":name,"count":s_rules[name],"percent":100*s_rules[name]/s_rows} for name in s_rule_names])
print("Proposed rules (overlapping):");print(s_rule_table.to_string(index=False))
print("Unique rows meeting any measured anomaly:",s_unique_anomalies,100*s_unique_anomalies/s_rows)
print("Unique rows excluded from proposed shared model view:",s_model_union,100*s_model_union/s_rows)
print("Retained shared model rows:",s_rows-s_model_union)
print("Raw rows removed: 0. Proposed numeric-value alterations: 0. No rules applied.")
print(pd.DataFrame(s_monthly).to_string(index=False))
assert s_snapshot=={str(p):(p.stat().st_size,p.stat().st_mtime_ns) for p in s_files}
print("PASS: source sizes and modification times unchanged; Step 6 investigation only.")
# Compact evidence for the decision report; aggregate values only.
s_evidence={"rows":s_rows,"groups":dict(s_sizes),"relationships":{g:dict(x) for g,x in s_rel.items()},"rules":dict(s_rules),"boundary":dict(s_boundary_counts),"model_union":s_model_union,"anomaly_union":s_unique_anomalies}
print("EVIDENCE_JSON="+json.dumps(s_evidence))


Pairwise anomaly overlap; diagonal is each anomaly count:
                       negative_fare  negative_distance  zero_distance_charged  zero_riders  nonpositive_duration  speed_over_100  historical  out_of_month
negative_fare                2400031                  0                 204636          345                   193             478           0          1023
negative_distance                  0                  0                      0            0                     0               0           0             0
zero_distance_charged         204636                  0                1471746         8555                 16619               0           0            74
zero_riders                      345                  0                   8555       231578                  2125             500           0            56
nonpositive_duration             193                  0                  16619         2125                651610               0           0             1
speed_

# Data cleaning decisions: Step 6

**Status: proposed policy, not applied.** Investigation covers all **48,601,782 rows** in 12 monthly files. Raw data and Steps 1-5 remain unchanged. No cleaned dataset, splits, models, imputation, or corrections were produced. Notebook: `notebooks/01_data_audit_and_cleaning.ipynb`.

## Decisions

| Anomaly | Count | Percent | Proposed Treatment | Justification |
|---|---:|---:|---|---|
| Negative base_fare | 2,400,031 | 4.93815433% | KEEP raw; FILTER FOR FARE MODELLING | Mixed signed-charge behavior, not verified refunds; exclude negative targets from a nonnegative gross-fare model. |
| Negative final charge (additional evidence) | 875,399 | 1.80116647% | KEEP raw; FILTER FOR FARE MODELLING | Includes 5,572 rows with nonnegative base fare. Combined base-or-final-negative rule affects 2,405,603 rows. |
| Negative distance | 0 | 0.00000000% | FILTER FOR DISTANCE/TIME MODELLING if present | No observed rows; negative physical distance would require review. |
| Zero distance with nonzero base fare | 1,471,746 | 3.02817292% | KEEP; FLAG FOR REVIEW | Waiting/short trips or incomplete distance readings are plausible; no blanket removal. |
| rider_count == 0 | 231,578 | 0.47648047% | KEEP; NO IMPUTATION | Likely provider-specific recording behavior; unknown/default is plausible but unproven. |
| Negative duration | 1,942 | 0.00399574% | KEEP raw; FILTER FOR TIME/SPEED MODELLING pending review | 1,431 repeated-hour candidates; do not automatically add an hour or delete trips. |
| Zero duration | 649,668 | 1.33671642% | KEEP raw; FILTER FOR TIME/SPEED MODELLING | 632,625 have positive distance. Elapsed time is not usable as recorded. |
| Speed >100 mph | 11,899 | 0.02448264% | FILTER FOR TIME/SPEED MODEL VIEW; retain raw | Proposed conservative ceiling. Retain 80-100 mph for review; no winsorization. |
| Historical pickup/dropoff 2008/2009 | 8 | 0.00001646% | QUARANTINE FROM ANALYSIS; retain raw | All eight full records conflict with the stated observation period; do not invent replacement years. |
| Other pickup outside observation interval | 6 | 0.00001235% | KEEP master; EXCLUDE FROM PERIOD-SCOPED VIEW ONLY | Four March 31, 2025 pickups and two April 1, 2026 pickups. Scope exclusions, not invalid trips. |
| Either timestamp outside source month | 19,646 | 0.04042239% | KEEP; partition and review | 19,251 short boundary examples; 374 other adjacent cases; 13 remaining mismatches; 8 historical. No blanket file-month rule. |

These are view-specific policies, not universal deletion rules. Keep an intact raw/master archive. Financial adjustments may remain useful for accounting, while unusable duration measurements should not enter duration/speed targets. Do not automatically exclude a valid trip from demand analysis solely because its fare is negative or a timestamp needs interpretation.

## Investigation supporting the decisions

### Negative base fares and final charges

All **2,400,031 negative-base-fare rows are provider code 2**. Payment code 0 accounts for **1,694,004 (70.58%)**; code 4 for **455,956 (19.00%)**; code 2 for **162,267 (6.76%)**; code 3 for **86,005 (3.58%)**; and code 1 for **1,799 (0.07%)**. Rate class and offline flag are missing in 1,694,004 of these rows. Detailed counts, within-group percentages, code-conditioned percentages, and payment/provider combinations are in the notebook.

Only **869,827 (36.24%)** have a negative final charge; **1,530,204 (63.76%)** have a positive final charge. Another monetary component is negative in **706,187 (29.42%)**. Most have positive duration: **2,399,838**; **2,195,395** have positive distance. Mean base fare is **-10.2942**, mean final charge **-6.5527**, and mean duration **18.1944 minutes**. Sample median duration is about **16 minutes**, versus **13.73** for nonnegative fares.

This is a mixture of signed financial records and apparently active trips, not evidence that all negatives are refunds or voids. No transaction identifiers or matched reversal evidence establish such a classification. The [TLC dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) describes flex-fare, dispute, no-charge, and other payment types in TLC data; **the competition mapping is unconfirmed**, so these names are hypotheses for the observed codes. Do not take absolute values, replace negative fares with zero, or infer code meanings as facts.

**Policy:** retain raw and accounting records; exclude `base_fare < 0 OR charge_total < 0` from a nonnegative gross-fare training view. This catches **2,405,603** rows, including **5,572** additional negative-final-charge rows. If the team instead models net signed charges, this exclusion is inappropriate and must be changed. Avoid applying it automatically to demand-only or duration-only tasks.

### Zero distance with nonzero fare

**1,455,127 (98.87%)** have positive duration; **405,252 (27.54%)** have equal origin/destination IDs. Equal IDs establish the same zone, not the same physical point. Mean duration is **15.0950 minutes**, sample median **12.9667**, and sample p95 **38.2842**. Mean base fare is **22.6166**, sample median **17.605**, and sample p95 **77.00**. Payment code 0 represents **67.93%**, provider 2 **89.78%**, and rate code 5 **10.95%**; rate code is missing for **67.93%**.

[TLC fare rules](https://www.nyc.gov/site/tlc/passengers/taxi-fare.page) permit time-based charging while waiting or moving slowly. This supports retaining possible waiting/short trips but does not explain every different-zone zero-distance record. **KEEP and flag; no distance imputation or blanket removal.** Distance-sensitive models need an explicit later decision about these measurements. Under the combined eligibility scenario below, **1,250,568** remain after other exclusions.

### Zero riders compared with positive riders

Provider 1 accounts for **215,567 (93.09%)** of zero-rider rows, versus **20.89%** of positive-rider rows. Payment code 1 accounts for **82.76%** versus **84.21%**. **222,846 (96.23%)** zero-rider rows have positive distance and **229,453 (99.08%)** positive duration.

| Measure | Zero riders | Positive riders |
|---|---:|---:|
| Mean base fare | 17.1411 | 19.3343 |
| Mean distance, miles | 2.5540 | 3.5770 |
| Mean duration, minutes | 14.5022 | 17.5126 |
| Approximate sample median duration, minutes | 11.30 | 12.775 |

Neither group has missing original source fields. This means zero-rider records do **not** simply reproduce the known broadly missing-field pattern. Provider concentration and otherwise trip-like values support a recording/default hypothesis, but do not prove that zero means unknown. **KEEP and flag; do not replace with 1, a median, or missing without confirmed semantics.** Exclude the rider-count field from passenger-count interpretation until clarified, rather than discarding otherwise useful trips. **228,606** such rows remain under the combined eligibility scenario.

### Negative and zero durations

There are **1,942 negative** and **649,668 zero** durations. Of the negative durations, **1,431 (73.69%)** fall on November 2, 2025 with both timestamps in the repeated 01:00 hour and a difference between -60 and 0 minutes. [NIST](https://www.nist.gov/pml/time-and-frequency-division/popular-links/daylight-saving-time-dst) documents the clock rollback on the first Sunday in November. This is compatible with a clock ambiguity **if** these are applicable local times, but the source is timezone-naive and does not prove that interpretation.

**632,625 zero-duration records have positive distance**, making their elapsed travel time unusable as recorded. **KEEP raw; exclude all 651,610 non-positive durations from duration/speed model eligibility until resolved.** Do not add an hour, invent a duration, or calculate speed for them. They need not be removed from unrelated accounting/demand views.

### Proposed speed ceiling: 100 mph

Use **strictly greater than 100 mph** as the proposed time/speed modelling exclusion. Values equal to 100 remain eligible. No capping or rewriting. Keep 80-100 mph for review.

The exact p99.9 is **46.8506 mph**; p99 is **34.2581 mph**, while the maximum is **4,816,614.8 mph**. [NY DMV guidance](https://dmv.ny.gov/brochure/mv21.pdf) describes expressways normally at 55 mph and some at 65 mph. A **100 mph trip average is about 54% above 65 mph and 2.13 times the observed p99.9**, leaving a large tolerance for short-trip timing/distance error and out-of-area travel. It is an operational modelling tolerance, not a legal limit for every route or a learned physical boundary.

Threshold sensitivity: >80 affects **13,589**, >100 **11,899**, and >120 **10,672** rows. Choosing 100 retains **1,690** rows in (80,100] while excluding **1,227** in (100,120] plus the extreme tail. Within (80,100], **488/1,690** last at most a minute; above 120, **6,512/10,672** do. Short-record instability and physically extreme samples support a generous ceiling rather than a percentile-only cutoff. The value is a judgement with explicit tolerance, **not justified merely by roundness**; team confirmation is still needed for target-specific use and out-of-area exceptions.

### Historical timestamps and month boundaries

All eight historical rows are printed with every original field. All are provider 2, have positive fares, and contain dates in 2008/2009. Seven have durations from roughly 0.17 to 68 minutes; one spans about 921.53 minutes. They may be real operational records with bad dates, but their years clearly conflict with the observation period. **Quarantine all eight from analysis views; retain raw and do not guess replacement dates.**

Of **19,646** rows with either timestamp outside its source month:

- **19,251** are nonhistorical short near-boundary cases (both timestamps within two hours of a source-month boundary, positive duration at most two hours). These are plausible boundary activity, not certified valid in every other field.
- **374** are other adjacent-boundary cases. Samples include roughly 18-24-hour durations for short distances; review separately without a blanket filename rule.
- **13** are remaining nonhistorical mismatches. Examples span several days for small distances, such as 0.49 miles over about 4,062 minutes; these look implausible as ordinary metered trips and need targeted adjudication.
- **8** are historical.

**Retain legitimate boundary crossings.** The two-hour grouping is an inspection convention, not a cleaning cutoff. Flag the 374 and 13 groups for review; no new maximum-duration rule is inferred from a file-month mismatch. Their final duration-specific treatment remains unresolved and is **not an additional exclusion in the count below**.

Define a proposed period-scoped view by pickup in **[2025-04-01, 2026-04-01)**, retaining dropoffs after midnight or after the period end when pickup is in scope. This scopes out the eight historical pickups and **six additional legitimate boundary pickups** (four March 31, two April 1). Keep those six in the master; excluding them from this interval is not cleaning them as bad records. Confirm the intended pickup-based period with the team.

## Exact overlap and affected-row accounting

| Proposed exclusion rule | Rows | Percent of all rows |
|---|---:|---:|
| Historical analysis quarantine | 8 | 0.00001646% |
| Other out-of-period pickups (scope only) | 6 | 0.00001235% |
| Negative base fare OR negative final charge | 2,405,603 | 4.94961893% |
| Negative distance | 0 | 0.00000000% |
| Non-positive duration | 651,610 | 1.34071216% |
| Speed >100 mph | 11,899 | 0.02448264% |


These rules overlap. The notebook contains the full pairwise anomaly matrix and monthly proposed union counts. Examples: **204,636** negative-fare rows also have zero distance/nonzero fare; **16,619** zero-distance charged rows have non-positive duration; **500** zero-rider rows exceed 100 mph; one historical row exceeds 100 mph. There are **4,554,053 (9.37013585%)** distinct rows meeting any measured anomaly (using >100 mph for the speed category), but many are intentionally retained.

For a conservative shared fare/time model view using the **OR of all six proposed exclusion rules**:

- **Unique rows excluded or affected by at least one proposed exclusion: 3,068,448 (6.31344752%).**
- **Remaining rows: 45,533,334.**
- Sum of individual rule counts is **3,069,126**; that is **678** larger than the distinct union and must not be used as the removed-row count.
- **Raw/master rows proposed for deletion: 0. Numeric values proposed for alteration/imputation: 0. Actual changes applied: 0.**

The 3,068,448 union is an explicit scenario, not a universal mandate for every task. Target-specific views should use the relevant subset; no exclusions should silently carry into demand counts or net-accounting analysis. The unresolved long-duration boundary review could change a future policy and would require recomputing its union. This report does not claim the remaining rows are fully clean.

## Team confirmations before implementation

1. Confirm competition code meanings and whether signed fares represent net accounting, adjustments, or a nonnegative gross-fare target.
2. Confirm task-specific model eligibility versus a single shared model view, including the 100 mph ceiling and route coverage.
3. Confirm timezone/clock conventions before any repeated-hour correction; the 1,431 candidates are not proven repairs.
4. Confirm zero-rider semantics; retain without imputation until documented.
5. Confirm pickup-based observation scope and adjudicate the 374 adjacent/13 other timestamp cases before treating the duration view as final.

No approval is being requested to perform further work here: this step stops at documented recommendations. Applying any rule is a later task.

## Method and verification

Counts, code percentages, missingness, means, extrema, overlaps, and union sizes are exact over all input rows. Subgroup medians/p95 are approximate from fixed-seed random-priority samples (up to 5,000 per group); full-dataset speed percentiles are the exact Step 5 results. Source file sizes and modification times remained unchanged. All Step 6 code completed without errors and Steps 1-5 were compared unchanged. No cleaned Parquet, splits, or models were created.


## Step 7: Build the cleaned analytical dataset

The user's Step 7 instruction authorizes applying the **shared fare/time modelling scenario** already documented in `reports/data_cleaning_decisions.md`. That report and Steps 1-6 remain unchanged as historical records. This output is a period-scoped shared model view, not a complete demand or net-accounting population.

Apply exactly the six documented exclusions: historical timestamps; other pickups outside `[2025-04-01, 2026-04-01)`; negative base fare or final charge; negative distance; non-positive duration; and speed strictly above 100 mph. Keep eligible zero distances, zero riders, and month-boundary trips. No capping, correction, imputation, or new duration cutoff is introduced. The unresolved boundary categories remain flagged.

Read/write in 200,000-row chunks. Keep every original column and value in retained rows, original timestamp text, all eight derived fields, source-row provenance, six readable zone fields, and relevant audit flags. Hash every raw file before and after. Validate each join as many-to-one and verify retained original values are unchanged. Write a temporary Parquet file, reopen and scan it, and publish the final filename only after validation.

In [7]:
from pathlib import Path
from collections import Counter
import hashlib,json,re,shutil
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

b_root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"data"/"raw").is_dir())
b_policy_path=b_root/"reports"/"data_cleaning_decisions.md"
b_policy_bytes=b_policy_path.read_bytes();b_policy=b_policy_bytes.decode("utf-8")
b_policy_sha=hashlib.sha256(b_policy_bytes).hexdigest()
b_expected_rules={"historical_quarantine":8,"outside_observation_nonhistorical":6,"negative_fare_or_total_model":2_405_603,"negative_distance_model":0,"nonpositive_duration_model":651_610,"speed_over_100_model":11_899}
# Fail on a changed policy instead of silently substituting undocumented rules.
for text in ["3,068,448","45,533,334","2,405,603","651,610","11,899","strictly greater than 100 mph","[2025-04-01, 2026-04-01)"]:
    assert text in b_policy, f"Review changed cleaning policy: missing {text}"
b_expected_input=48_601_782;b_expected_retained=45_533_334;b_expected_excluded=3_068_448
b_raw=b_root/"data"/"raw"
b_all_raw=sorted(p for p in b_raw.rglob("*") if p.is_file())
b_files=sorted(p for p in b_all_raw if p.name.startswith("Urban_Flow_Analytics_Taxi_Dataset_") and p.suffix.lower()==".csv" and "__MACOSX" not in p.parts and not any(part.startswith("._") for part in p.parts))
b_zones=[p for p in b_all_raw if p.suffix.lower()==".csv" and "zone" in p.stem.lower() and "__MACOSX" not in p.parts and not any(part.startswith("._") for part in p.parts)]
assert len(b_files)==12 and len(b_zones)==1
b_output=b_root/"data"/"processed"/"clean_trips.parquet"
b_partial=b_output.with_name("clean_trips.parquet.partial")
if b_output.exists():
    print("Processed dataset already exists; skipping rebuild and continuing with verification.")
else:
    assert not b_partial.exists(), "Partial build exists; review before replacing"
b_output.parent.mkdir(parents=True,exist_ok=True)
print("Policy SHA-256:",b_policy_sha)
print("Free disk GB:",shutil.disk_usage(b_output.parent).free/1e9)
def b_sha(path):
    digest=hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b""):digest.update(block)
    return digest.hexdigest()
print("Hashing raw files before processing",flush=True)
b_before={str(p.relative_to(b_root)): {"bytes":p.stat().st_size,"sha256":b_sha(p)} for p in b_all_raw}
print(json.dumps(b_before,indent=2))
b_zone=pd.read_csv(b_zones[0])
assert b_zone.loc_id.notna().all() and b_zone.loc_id.is_unique
b_zone_labels=["borough_name","zone_name","service_zone"]
assert all(col in b_zone for col in b_zone_labels)
b_pickup_lookup=b_zone.rename(columns={"loc_id":"origin_loc_id",**{col:"pickup_"+col for col in b_zone_labels}})
b_dropoff_lookup=b_zone.rename(columns={"loc_id":"dest_loc_id",**{col:"dropoff_"+col for col in b_zone_labels}})
b_original=pd.read_csv(b_files[0],nrows=0).columns.tolist()
for p in b_files:assert pd.read_csv(p,nrows=0).columns.tolist()==b_original
# Preserve source numeric widths; compact only derived calendar and Boolean columns.
b_ints=["provider_code","origin_loc_id","dest_loc_id","fare_settlement_method"]
b_strings=["pickup_timestamp","dropoff_timestamp","offline_record_flag"]
b_read_dtypes={col:"int64" if col in b_ints else "string" if col in b_strings else "float64" for col in b_original}

Processed dataset already exists; skipping rebuild and continuing with verification.
Policy SHA-256: 9e9938db705620460a8fd5f5c12fceef489c37fb535f77f255134296f079b43c
Free disk GB: 1.398714368
Hashing raw files before processing
{
  "data\\raw\\.gitkeep": {
    "bytes": 0,
    "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
  },
  "data\\raw\\Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv": {
    "bytes": 429662745,
    "sha256": "dba50280078b3316ab6bf8fefb539e4b9d8c9f580c5f514ec4e9599165d503fd"
  },
  "data\\raw\\Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv": {
    "bytes": 492535780,
    "sha256": "acc84c0f3aa8cc7cff779e8c359bad20ec88ffc9d9be08fdb8e42902fe65a2df"
  },
  "data\\raw\\Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv": {
    "bytes": 462646895,
    "sha256": "a567e88093ad1feffd75d2b8f2422054fe76927ed57ea16e4a72f4fe6824ac74"
  },
  "data\\raw\\Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv": {
    "bytes": 418006505,
    "sha256": "4c57cbc07f4cc3c4

In [4]:
from pathlib import Path
import pyarrow.parquet as pq

output_file = Path("../data/processed/clean_trips.parquet")

print("Exists:", output_file.exists())

if output_file.exists():
    parquet_file = pq.ParquetFile(output_file)

    print("Rows:", parquet_file.metadata.num_rows)
    print("Columns:", parquet_file.metadata.num_columns)
    print("Row groups:", parquet_file.metadata.num_row_groups)
    print("Size MB:", output_file.stat().st_size / (1024 ** 2))

Exists: True
Rows: 45533334
Columns: 45
Row groups: 247
Size MB: 1218.7746315002441


In [6]:
from pathlib import Path
import pyarrow.parquet as pq

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "processed").is_dir())
output_file = root / "data" / "processed" / "clean_trips.parquet"
parquet_file = pq.ParquetFile(output_file)

print("Total rows:", parquet_file.metadata.num_rows)
print("Total columns:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.metadata.num_row_groups)
print("File size (MB):", output_file.stat().st_size / 1_000_000)

assert parquet_file.metadata.num_row_groups > 0, "Parquet file has no row groups"
first_row_group = parquet_file.read_row_group(0)
sample = first_row_group.to_pandas().head(1000)

print("Sample shape:", sample.shape)
print("All sample columns:")
print(sample.columns.tolist())

required_fields = [
    "trip_duration_minutes", "speed_mph", "pickup_date",
    "pickup_hour", "day_of_week", "month", "weekend", "route_id",
]
print("Required fields present:")
print({field: field in sample.columns for field in required_fields})

print("Zone/borough columns:")
print([column for column in sample.columns if "zone" in column.lower() or "borough" in column.lower()])

flag_columns = [column for column in sample.columns if column.startswith("flag_")]
print("Flag columns:")
print(flag_columns)
print("Flag counts within sample:")
for column in flag_columns:
    print(f"{column}:")
    print(sample[column].value_counts(dropna=False).to_dict())

print("STOP: Step 7 verification only; no data was modified and Step 8 was not run.")

Total rows: 45533334
Total columns: 45
Row groups: 247
File size (MB): 1277.977828
Sample shape: (1000, 45)
All sample columns:
['provider_code', 'pickup_timestamp', 'dropoff_timestamp', 'rider_count', 'distance_miles', 'rate_class_id', 'offline_record_flag', 'origin_loc_id', 'dest_loc_id', 'fare_settlement_method', 'base_fare', 'surcharge_misc', 'transit_tax', 'driver_tip_payment', 'toll_total', 'service_improvement_fee', 'charge_total', 'zone_congestion_fee', 'Airport_fee', 'congestion_relief_fee', 'source_file', 'source_month', 'source_row_1based', 'trip_duration_minutes', 'speed_mph', 'pickup_date', 'pickup_hour', 'day_of_week', 'month', 'weekend', 'route_id', 'audit_zero_distance_nonzero_fare', 'audit_zero_riders', 'audit_speed_80_to_100', 'audit_pickup_outside_source_month', 'audit_dropoff_outside_source_month', 'audit_boundary_category', 'pickup_borough_name', 'pickup_zone_name', 'pickup_service_zone', 'dropoff_borough_name', 'dropoff_zone_name', 'dropoff_service_zone', 'audit_p

## Step 9: Chronological train, validation, and test splits

The processed dataset was split incrementally by `pickup_timestamp` using PyArrow row groups. The full dataset was not loaded into pandas, rows were not shuffled, and `data/processed/clean_trips.parquet` and `data/feature_contract.md` were not modified.

### Pickup coverage and exact boundaries

- Overall pickup coverage: `2025-04-01 00:00:00` through `2026-03-31 23:59:59`.
- Train: timestamps `< 2025-12-11 00:00:00`.
- Validation: timestamps `>= 2025-12-11 00:00:00` and `< 2026-02-05 00:00:00`.
- Test: timestamps `>= 2026-02-05 00:00:00`.
- Boundaries are whole-day cut points selected from cumulative pickup-date counts near 70% and 85%; therefore the proportions are approximate rather than exact row percentages.

| Split | Rows | Percent | Start Timestamp | End Timestamp | File Size |
|---|---:|---:|---|---|---:|
| Train | 31,988,176 | 70.252216% | 2025-04-01 00:00:00 | 2025-12-10 23:59:59 | 894.731 MB |
| Validation | 6,814,901 | 14.966839% | 2025-12-11 00:00:00 | 2026-02-04 23:59:58 | 190.425 MB |
| Test | 6,730,257 | 14.780945% | 2026-02-05 00:00:01 | 2026-03-31 23:59:59 | 186.475 MB |

### Verification

- Output files: `data/splits/train.parquet`, `data/splits/validation.parquet`, and `data/splits/test.parquet`.
- Total row reconciliation: `31,988,176 + 6,814,901 + 6,730,257 = 45,533,334`; **PASS**.
- No rows were lost: **PASS**, based on one exclusive timestamp mask per source row and total-count reconciliation.
- Duplicate rows between splits: **none detected**; the exclusive, non-overlapping timestamp masks assign each source row to at most one split.
- Timestamp range overlap: **none**; train ends before validation starts, and validation ends before test starts.
- Latest-period test check: **PASS**; test contains the latest observed pickup timestamp, `2026-03-31 23:59:59`.
- Warning: files use compact Zstandard Parquet compression because the source and split files are large. The split contents retain all source columns and row values; physical row order from incremental source processing is preserved, with no random shuffle.

**STOP: Step 9 complete. No modelling or later-step processing was performed.**